# Chapter 10: Memory

> *“Memory is like an open field. If you do not erect fences and
> establish clear ownership, weeds will overrun the soil, and cattle
> will trample the crops.”*  
> — **Systems Proverb**

------------------------------------------------------------------------

In Chapter 9, we examined the mechanical anatomy of the machine: how the
Central Processing Unit retrieves binary words over the physical bus,
alters register states, and drives calculations. We saw that primary
memory (RAM) is simply a contiguous array of byte-sized storage cells,
indexed from address `0x00000000` to the upper physical limits of the
hardware.

In high-level languages such as Python, memory is an ambient utility.
You create objects, append elements to arrays, and form interconnected
graphs of data without ever considering where those bits reside in
physical hardware. A runtime engine continuously tracks references
behind the scenes, periodically running a **garbage collector** to sweep
discarded objects from the heap.

In Modern C++, we step out from behind the safety of the garbage
collector. We take direct command of memory.

Direct memory control gives C++ its speed, predictable latency, and
compact footprint. However, with that power comes absolute
responsibility. A single misplaced byte offset, uninitialised pointer,
or dangling reference can corrupt execution or introduce catastrophic
security vulnerabilities.

In this chapter, we will demystify memory. We will dissect the process
address space, contrast the deterministic stack against the dynamic
heap, master pointers and references, understand structural alignment,
and explore the crown jewel of C++ systems architecture: **Resource
Acquisition Is Initialisation (RAII)** and **Smart Pointers**. Finally,
we will implement custom dynamic memory structures from scratch.

### 10.1 The Virtual Memory Architecture and Process Layout

When an operating system (such as Linux, macOS, or Windows) launches a
compiled C++ binary, it does not assign the program a direct, unmediated
slice of physical RAM chips. If every program wrote directly to physical
memory addresses, one misbehaving process could overwrite the operating
system kernel or steal cryptographic keys from another application.

Instead, the CPU’s **Memory Management Unit (MMU)** works with the
operating system kernel to present every process with an illusion: a
private, contiguous, uniform address range known as **Virtual Memory**.

On a standard 64-bit architecture ($x86\text{-}64$ or $\text{ARM}64$),
this virtual address space is astronomically vast ($2^{64}$ bytes,
though current hardware typically implements a 48-bit or 52-bit virtual
addressing range, yielding 256 Terabytes of addressable space per
process). The operating system maps these virtual addresses to physical
RAM frames via **Page Tables**.

When your compiled C++ program loads into virtual memory, its address
space is organised into distinct semantic segments (see fig. 1):

<figure id="fig:address-space">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIwIDAgOTAwIDYyMCIgd2lkdGg9IjEwMCUiIGhlaWdodD0iNjIwIiB4bWxu
cz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciPgogIDxkZWZzPgogICAgPHN0eWxlPgogICAg
ICA6cm9vdCB7CiAgICAgICAgLS1zdmctYmc6IHZhcigtLWJnLCAjZmZmZmZmKTsKICAgICAgICAt
LXN2Zy1mZzogdmFyKC0tZmcsICMwMDAwMDApOwogICAgICAgIC0tc3ZnLXRpbnQtMS1mYWxsYmFj
azogI2NjY2NjYzsgLyogMjAlIGJsYWNrICsgODAlIHdoaXRlICovCiAgICAgICAgLS1zdmctdGlu
dC0yLWZhbGxiYWNrOiAjOTk5OTk5OyAvKiA0MCUgYmxhY2sgKyA2MCUgd2hpdGUgKi8KICAgICAg
ICAtLXN2Zy10aW50LTE6IGNvbG9yLW1peChpbiBzcmdiLCB2YXIoLS1zdmctZmcpIDIwJSwgdmFy
KC0tc3ZnLWJnKSA4MCUpOwogICAgICAgIC0tc3ZnLXRpbnQtMjogY29sb3ItbWl4KGluIHNyZ2Is
IHZhcigtLXN2Zy1mZykgNDAlLCB2YXIoLS1zdmctYmcpIDYwJSk7CiAgICAgIH0KICAgICAgdGV4
dCAgIHsgZm9udC1mYW1pbHk6ICJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIsIG1vbm9zcGFj
ZTsgfQogICAgPC9zdHlsZT4KICAgIDxmaWx0ZXIgaWQ9Im1lbS1zaGFkb3ciIHg9Ii01JSIgeT0i
LTUlIiB3aWR0aD0iMTEwJSIgaGVpZ2h0PSIxMTAlIj4KICAgICAgPGZlRHJvcFNoYWRvdyBkeD0i
MiIgZHk9IjIiIHN0ZERldmlhdGlvbj0iMyIgZmxvb2Qtb3BhY2l0eT0iMC4xNSIvPgogICAgPC9m
aWx0ZXI+CiAgICA8IS0tIEFycm93aGVhZCB0YXJnZXRpbmcgbGluZSBlbmQgKHBvaW50aW5nIGRv
d24pIC0tPgogICAgPG1hcmtlciBpZD0ibWVtLWFycm93LWRvd24iIHZpZXdCb3g9IjAgMCAxMCAx
MCIgcmVmWD0iNSIgcmVmWT0iNyIgbWFya2VyV2lkdGg9IjYiIG1hcmtlckhlaWdodD0iNiIgb3Jp
ZW50PSIwIj4KICAgICAgPHBhdGggZD0iTSAwIDAgTCAxMCAwIEwgNSA3IHoiIGZpbGw9ImNvbnRl
eHQtc3Ryb2tlIiAvPgogICAgPC9tYXJrZXI+CiAgICA8IS0tIEFycm93aGVhZCB0YXJnZXRpbmcg
bGluZSBlbmQgKHBvaW50aW5nIHVwKSAtLT4KICAgIDxtYXJrZXIgaWQ9Im1lbS1hcnJvdy11cCIg
dmlld0JveD0iMCAwIDEwIDEwIiByZWZYPSI1IiByZWZZPSIzIiBtYXJrZXJXaWR0aD0iNiIgbWFy
a2VySGVpZ2h0PSI2IiBvcmllbnQ9IjAiPgogICAgICA8cGF0aCBkPSJNIDAgNyBMIDEwIDcgTCA1
IDAgeiIgZmlsbD0iY29udGV4dC1zdHJva2UiIC8+CiAgICA8L21hcmtlcj4KICA8L2RlZnM+CiAg
PCEtLSBQcm9jZXNzIE1lbW9yeSBPdXRlciBGcmFtZSAtLT4KICA8cmVjdCB4PSIxODAiIHk9IjMw
IiB3aWR0aD0iNTYwIiBoZWlnaHQ9IjU2MCIgcng9IjEwIiBmaWxsPSJ2YXIoLS1zdmctdGludC0y
LCB2YXIoLS1zdmctdGludC0yLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ry
b2tlLXdpZHRoPSIyIiBmaWx0ZXI9InVybCgjbWVtLXNoYWRvdykiLz4KICAgIDwhLS0gQWRkcmVz
cyBMYWJlbHMgKExlZnQgU2lkZSkgLS0+CiAgPHRleHQgeD0iMzUiIHk9IjU1IiBmb250LXNpemU9
IjE0IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+MHhGRkZGRkZGRkZG
RkZGRkZGPC90ZXh0PgogIDx0ZXh0IHg9IjM1IiB5PSIxMDUiIGZvbnQtc2l6ZT0iMTQiIGZvbnQt
d2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIj4weDdGRkZGRkZGRkZGRjwvdGV4dD4K
ICA8dGV4dCB4PSIzNSIgeT0iMjQwIiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIg
ZmlsbD0idmFyKC0tc3ZnLWZnKSI+MHg3RkZGMDAwMDAwMDA8L3RleHQ+CiAgPHRleHQgeD0iMzUi
IHk9IjQyMCIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2
Zy1mZykiPjB4MDAwMDU1NTUwMDAwPC90ZXh0PgogIDx0ZXh0IHg9IjM1IiB5PSI1NzUiIGZvbnQt
c2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIj4weDAwMDAw
MDAwMDAwMDwvdGV4dD4KICA8IS0tIDEuIEtlcm5lbCBTcGFjZSAtLT4KICA8cmVjdCB4PSIxOTAi
IHk9IjQwIiB3aWR0aD0iNTQwIiBoZWlnaHQ9IjU1IiByeD0iNiIgZmlsbD0idmFyKC0tc3ZnLXRp
bnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZyki
IHN0cm9rZS13aWR0aD0iMS41Ii8+CiAgPHRleHQgeD0iNDYwIiB5PSI2MyIgZm9udC1zaXplPSIx
NCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiIHRleHQtYW5jaG9yPSJt
aWRkbGUiPktlcm5lbCBTcGFjZTwvdGV4dD4KICA8dGV4dCB4PSI0NjAiIHk9IjgxIiBmb250LXNp
emU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiB0ZXh0LWFuY2hvcj0ibWlkZGxlIj4oUmVzZXJ2
ZWQgZm9yIE9TIENvcmUsIFBhZ2UgVGFibGVzICZhbXA7IERldmljZSBEcml2ZXJzIC0gSW5hY2Nl
c3NpYmxlIHRvIFVzZXIpPC90ZXh0PgogIDwhLS0gMi4gU3RhY2sgLS0+CiAgPHJlY3QgeD0iMTkw
IiB5PSIxMDUiIHdpZHRoPSI1NDAiIGhlaWdodD0iODUiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmct
dGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZn
KSIgc3Ryb2tlLXdpZHRoPSIxLjUiLz4KICA8dGV4dCB4PSI0NjAiIHk9IjEzMCIgZm9udC1zaXpl
PSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiIHRleHQtYW5jaG9y
PSJtaWRkbGUiPlRoZSBTdGFjazwvdGV4dD4KICA8dGV4dCB4PSI0NjAiIHk9IjE1MCIgZm9udC1z
aXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgdGV4dC1hbmNob3I9Im1pZGRsZSI+TG9jYWwg
dmFyaWFibGVzLCBmdW5jdGlvbiBjYWxsIGZyYW1lcywgcmV0dXJuIGFkZHJlc3NlcywgcGFyYW1l
dGVyIGJpbmRpbmdzPC90ZXh0PgogIDx0ZXh0IHg9IjQ2MCIgeT0iMTcyIiBmb250LXNpemU9IjE0
IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgdGV4dC1hbmNob3I9Im1p
ZGRsZSI+R3Jvd3MgRG93bndhcmRzICYjODU5NTs8L3RleHQ+CiAgPCEtLSBTdGFjayBHcm93dGgg
QXJyb3cgLS0+CiAgPHBhdGggZD0iTSA0NTAgMTk1IEwgNDUwIDIzMCIgc3Ryb2tlPSJ2YXIoLS1z
dmctZmcpIiBzdHJva2Utd2lkdGg9IjMiIG1hcmtlci1lbmQ9InVybCgjbWVtLWFycm93LWRvd24p
Ii8+CiAgPCEtLSBVbmFsbG9jYXRlZCAvIEd1YXJkIFNwYWNlIC0tPgogIDxyZWN0IHg9IjE5MCIg
eT0iMjQwIiB3aWR0aD0iNTQwIiBoZWlnaHQ9IjcwIiByeD0iNiIgZmlsbD0idmFyKC0tc3ZnLXRp
bnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZyki
IHN0cm9rZS1kYXNoYXJyYXk9IjQsNCIvPgogIDx0ZXh0IHg9IjQ2MCIgeT0iMjcyIiBmb250LXNp
emU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgdGV4dC1hbmNo
b3I9Im1pZGRsZSI+VW5hbGxvY2F0ZWQgU3BhY2U8L3RleHQ+CiAgPHRleHQgeD0iNDYwIiB5PSIy
OTIiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiIHRleHQtYW5jaG9yPSJtaWRk
bGUiPihNZW1vcnktbWFwcGVkIGZpbGVzLCBTaGFyZWQgTGlicmFyaWVzKTwvdGV4dD4KICA8IS0t
IEhlYXAgR3Jvd3RoIEFycm93IC0tPgogIDxwYXRoIGQ9Ik0gNDUwIDM1NSBMIDQ1MCAzMjAiIHN0
cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIzIiBtYXJrZXItZW5kPSJ1cmwoI21l
bS1hcnJvdy11cCkiLz4KICA8IS0tIDMuIEhlYXAgLS0+CiAgPHJlY3QgeD0iMTkwIiB5PSIzNjAi
IHdpZHRoPSI1NDAiIGhlaWdodD0iODUiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2
YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tl
LXdpZHRoPSIxLjUiLz4KICA8dGV4dCB4PSI0NjAiIHk9IjM4MyIgZm9udC1zaXplPSIxNCIgZm9u
dC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiIHRleHQtYW5jaG9yPSJtaWRkbGUi
Pkdyb3dzIFVwd2FyZHMgJiM4NTkzOzwvdGV4dD4KICA8dGV4dCB4PSI0NjAiIHk9IjQwNSIgZm9u
dC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiIHRleHQt
YW5jaG9yPSJtaWRkbGUiPlRoZSBIZWFwPC90ZXh0PgogIDx0ZXh0IHg9IjQ2MCIgeT0iNDI1IiBm
b250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiB0ZXh0LWFuY2hvcj0ibWlkZGxlIj5E
eW5hbWljIG1hbnVhbCBhbGxvY2F0aW9ucyAobWFsbG9jIC8gbmV3IC8gc3RkOjp2ZWN0b3IgLyBz
bWFydCBwb2ludGVycyk8L3RleHQ+CiAgPCEtLSA0LiBCU1MgU2VnbWVudCAtLT4KICA8cmVjdCB4
PSIxOTAiIHk9IjQ1NSIgd2lkdGg9IjU0MCIgaGVpZ2h0PSIzNSIgcng9IjQiIGZpbGw9InZhcigt
LXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1z
dmctZmcpIiBzdHJva2Utd2lkdGg9IjEuNSIvPgogIDx0ZXh0IHg9IjQ2MCIgeT0iNDc3IiBmb250
LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgdGV4dC1h
bmNob3I9Im1pZGRsZSI+QlNTIFNlZ21lbnQgKFVuaW5pdGlhbGlzZWQgR2xvYmFscyAmYW1wOyBT
dGF0aWNzIC0gWmVyb2VkIGJ5IE9TKTwvdGV4dD4KICA8IS0tIDUuIERhdGEgU2VnbWVudCAtLT4K
ICA8cmVjdCB4PSIxOTAiIHk9IjQ5NSIgd2lkdGg9IjU0MCIgaGVpZ2h0PSIzNSIgcng9IjQiIGZp
bGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tl
PSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjEuNSIvPgogIDx0ZXh0IHg9IjQ2MCIgeT0i
NTE3IiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZn
KSIgdGV4dC1hbmNob3I9Im1pZGRsZSI+RGF0YSBTZWdtZW50IChJbml0aWFsaXNlZCBHbG9iYWwg
YW5kIFN0YXRpYyBWYXJpYWJsZXMpPC90ZXh0PgogIDwhLS0gNi4gVGV4dCAvIENvZGUgU2VnbWVu
dCAtLT4KICA8cmVjdCB4PSIxOTAiIHk9IjUzNSIgd2lkdGg9IjU0MCIgaGVpZ2h0PSI0NSIgcng9
IjQiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIg
c3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjEuNSIvPgogIDx0ZXh0IHg9IjQ2
MCIgeT0iNTU1IiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0t
c3ZnLWZnKSIgdGV4dC1hbmNob3I9Im1pZGRsZSI+VGV4dCAvIENvZGUgU2VnbWVudDwvdGV4dD4K
ICA8dGV4dCB4PSI0NjAiIHk9IjU3MCIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZn
KSIgdGV4dC1hbmNob3I9Im1pZGRsZSI+KENvbXBpbGVkIEJpbmFyeSBNYWNoaW5lIEluc3RydWN0
aW9ucyAtIFJlYWQgT25seSk8L3RleHQ+Cjwvc3ZnPg==
" alt="Figure 1: Virtual Memory Layout (Address Space)" />
<figcaption>Figure 1: Virtual Memory Layout (Address Space)</figcaption>
</figure>

#### 10.1.1 The Anatomy of Segments

- **Text (Code) Segment:** Contains the read-only executable machine
  instructions compiled directly from your C++ functions. Attempting to
  write to this memory causes an immediate hardware memory protection
  fault (`Segmentation Fault`).
- **Data Segment:** Contains global variables and `static` local
  variables that have explicit, non-zero initial values. These are
  allocated when the process initialises and persist for the entire
  runtime of the program.
- **BSS Segment (*Block Started by Symbol*):** Contains global and
  static variables that are uninitialised (or initialised to zero). The
  operating system does not store zeroes in the binary executable file;
  instead, it notes the required size and automatically maps these pages
  to zero-filled RAM on launch.
- **The Heap:** A dynamically managed memory reservoir. When data
  structures must grow dynamically at runtime (such as when parsing an
  unknown quantity of network packets or user inputs), space is
  requested on the heap. The heap starts at lower memory addresses and
  expands upwards towards higher addresses.
- **The Stack:** A strictly ordered, high-speed memory structure that
  manages function execution frames, local variables, and return
  targets. On modern hardware, the stack starts at high memory addresses
  and grows downwards towards lower addresses.

### 10.2 The Physical Stack versus The Dynamic Heap

Every single variable instantiated in a C++ program has a distinct
**Storage Duration** tied directly to the physical structure holding it.
The two most critical regions you will interact with are the **Stack**
and the **Heap**.

#### 10.2.1 The Call Stack: Automatic Storage Duration

The stack is a Last-In, First-Out (LIFO) structure managed directly by
the hardware register `RSP` (the Stack Pointer).

When a function is called, a **Stack Frame** (or *Activation Record*) is
allocated in a single CPU operation: the stack pointer is decremented by
the total byte size required for all local variables in that function.
When the function returns, the stack pointer is simply incremented back
to its previous position.

``` cpp
void compute(int factor) {
    int local_a = 10;          // Resides on stack frame
    double local_b = 3.14159;  // Resides on stack frame
    // [local_a and local_b live strictly within this scope]
} // Stack Frame popped automatically. Instantaneous deallocation.
```

#### Why the Stack is Blazingly Fast:

1.  **Zero Allocation Latency:** Allocating 100 bytes on the stack takes
    one CPU instruction: `sub rsp, 100`. Deallocation takes one
    instruction: `add rsp, 100`.
2.  **Predictable LIFO Lifetimes:** Destruction of objects occurs
    deterministically in exact reverse order of construction.
3.  **High Cache Locality:** Because stack frames reside in a tight,
    contiguous memory band, stack memory almost always resides inside
    the ultra-fast L1/L2 CPU hardware caches.

#### The Stack’s Fatal Limitation: Size

The call stack is finite. On Linux, the default thread stack is
typically 8 Megabytes; on Windows, it is often 1 Megabyte. If you
allocate a massive array on the stack:

``` cpp
void catastrophic_stack_overflow() {
    int massive_array[10'000'000]; // 40 Megabytes!
    massive_array[0] = 42;         // CRASH: Segmentation fault (Stack Overflow)
}
```

The stack pointer exceeds its allocated page boundaries, hitting a
protected “guard page”, and the operating system immediately terminates
the process.

#### 10.2.2 The Heap: Dynamic Storage Duration

When you do not know the size of data at compile time, or when you
require an object to survive beyond the function that created it, you
must allocate memory on the **Heap**.

The heap is not a single hardware-managed pointer; it is a complex,
user-space dynamic memory allocator (such as `ptmalloc`, `tcmalloc`, or
`jemalloc`) operating over pages requested from the operating system
kernel via system calls (`brk`, `sbrk`, or `mmap`).

``` cpp
void create_dynamic_buffer() {
    // Request 40 Megabytes on the Dynamic Heap
    int* dynamic_array = new int[10'000'000]; 
    
    dynamic_array[0] = 42; // Perfectly safe. Heap has gigabytes of capacity.

    delete[] dynamic_array; // Mandatory manual deallocation!
}
```

Usage of the stack and the heap is compared in tbl. 1.

| Architectural Trait | The Stack | The Heap |
|:-----------------------|:-----------------------|:-----------------------|
| **Allocation Mechanism** | CPU decrementing Stack Pointer (`RSP`) | Runtime Allocator searching free-lists / heaps |
| **Speed** | Instantaneous ($\approx 1\text{ CPU Cycle}$) | Slower ($\approx 20\text{--}100+\text{ Cycles}$) |
| **Lifetime Model** | Automatic (bounded strictly to enclosing `{}`) | Dynamic (persists until explicitly destroyed) |
| **Size Capacity** | Highly constrained ($1\text{--}8\text{ MB}$) | Bound only by physical RAM & swap space |
| **Fragmentation** | Zero fragmentation | Susceptible to external & internal fragmentation |
| **Access Pattern** | Contiguous, optimal CPU cache hits | Scattered pointers, potential CPU cache misses |

Table 1: Comparison Matrix: Stack vs Heap {#tbl:stack-vs-heap}

### 10.3 Pointers, Addresses, and Dereferencing

To manipulate memory, we must manipulate memory addresses. A **Pointer**
is nothing more than a variable whose value is the numeric memory
address of another object (as shown in fig. 2).

<figure id="fig:pointer">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI2MCA0MCA3NDAgMTkwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIxOTAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiwgbW9ub3Nw
YWNlOyB9CiAgICA8L3N0eWxlPgogICAgPGZpbHRlciBpZD0icHRyLXNoYWRvdyIgeD0iLTUlIiB5
PSItNSUiIHdpZHRoPSIxMTAlIiBoZWlnaHQ9IjExMCUiPgogICAgICA8ZmVEcm9wU2hhZG93IGR4
PSIyIiBkeT0iMiIgc3RkRGV2aWF0aW9uPSIzIiBmbG9vZC1vcGFjaXR5PSIwLjE1Ii8+CiAgICA8
L2ZpbHRlcj4KICAgIDxtYXJrZXIgaWQ9InB0ci1hcnJvdyIgdmlld0JveD0iMCAwIDEwIDEwIiBy
ZWZYPSI2IiByZWZZPSI1IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9
ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwYXRoIGQ9Ik0gMCAxIEwgMTAgNSBMIDAgOSB6
IiBmaWxsPSJjb250ZXh0LXN0cm9rZSIvPgogICAgPC9tYXJrZXI+CiAgPC9kZWZzPgogIDwhLS0g
VmFyaWFibGU6IFZhbHVlIEJveCAtLT4KICA8ZyB0cmFuc2Zvcm09InRyYW5zbGF0ZSg4MCwgNTAp
IiBmaWx0ZXI9InVybCgjcHRyLXNoYWRvdykiPgogICAgPHJlY3Qgd2lkdGg9IjI0MCIgaGVpZ2h0
PSIxNTAiIHJ4PSI4IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZh
bGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIxLjUiLz4KICAg
IDx0ZXh0IHg9IjE1IiB5PSIyNCIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZp
bGw9InZhcigtLXN2Zy1mZykiPlZhcmlhYmxlOiB0YXJnZXQgKGludCk8L3RleHQ+CiAgICA8dGV4
dCB4PSIxNSIgeT0iNjUiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPkFkZHJl
c3M6IDB4N0ZGRTAwMTg8L3RleHQ+CiAgICA8cmVjdCB4PSIxNSIgeT0iODAiIHdpZHRoPSIyMTAi
IGhlaWdodD0iNTAiIHJ4PSI0IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGlu
dC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIvPgogICAgPHRleHQgeD0iOTUi
IHk9IjExMiIgZm9udC1zaXplPSIxOCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2
Zy1mZykiPjQyPC90ZXh0PgogIDwvZz4KICA8IS0tIFBvaW50ZXIgVmFyaWFibGUgQm94IC0tPgog
IDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDUyMCwgNTApIiBmaWx0ZXI9InVybCgjcHRyLXNoYWRv
dykiPgogICAgPHJlY3Qgd2lkdGg9IjI2MCIgaGVpZ2h0PSIxNTAiIHJ4PSI4IiBmaWxsPSJ2YXIo
LS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0t
c3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIxLjUiLz4KICAgIDx0ZXh0IHg9IjE1IiB5PSIyNCIgZm9u
dC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiPlBvaW50
ZXI6IHB0ciAoaW50Kik8L3RleHQ+CiAgICA8dGV4dCB4PSIxNSIgeT0iNjUiIGZvbnQtc2l6ZT0i
MTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPkFkZHJlc3M6IDB4N0ZGRTAwMjA8L3RleHQ+CiAgICA8
cmVjdCB4PSIxNSIgeT0iODAiIHdpZHRoPSIyMzAiIGhlaWdodD0iNTAiIHJ4PSI0IiBmaWxsPSJ2
YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFy
KC0tc3ZnLWZnKSIvPgogICAgPHRleHQgeD0iNDUiIHk9IjExMiIgZm9udC1zaXplPSIxNiIgZm9u
dC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiPjB4N0ZGRTAwMTg8L3RleHQ+CiAg
PC9nPgogIDwhLS0gUG9pbnRlciBBcnJvdyBjb25uZWN0aW5nIHB0ciB2YWx1ZSB0byB0YXJnZXQg
YWRkcmVzcyAtLT4KICA8cGF0aCBkPSJNIDUzNSAxNTUgQyA0NDAgMjEwLCA0MDAgMjEwLCAzMjUg
MTU1IiBmaWxsPSJub25lIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMyIg
c3Ryb2tlLWRhc2hhcnJheT0iMiwyIiBtYXJrZXItZW5kPSJ1cmwoI3B0ci1hcnJvdykiLz4KICA8
dGV4dCB4PSIzNzUiIHk9IjIyMCIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZp
bGw9InZhcigtLXN2Zy1mZykiPlBvaW50cyBUbyAoJmFtcDt0YXJnZXQpPC90ZXh0Pgo8L3N2Zz4=
" alt="Figure 2: Pointer: Variable Holding an Address" />
<figcaption>Figure 2: Pointer: Variable Holding an Address</figcaption>
</figure>

#### 10.3.1 The Operators of Indirection

In C++, pointer syntax revolves around two fundamental unary operators:

1.  **The Address-Of Operator (`&`):** Yields the memory address of an
    existing object.
2.  **The Dereference Operator (`*`):** Accesses the memory cell
    situated at the address stored within the pointer.

``` cpp
#include <iostream>
#include <format>

int main() {
    int target = 42;
    int* ptr = &target; // ptr stores the address of target (e.g. 0x7FFE0018)

    std::cout << std::format("Value of target        : {}\n", target);
    std::cout << std::format("Address of target (&)  : {}\n", static_cast<void*>(&target));
    std::cout << std::format("Value held inside ptr  : {}\n", static_cast<void*>(ptr));
    std::cout << std::format("Address of ptr itself  : {}\n", static_cast<void*>(&ptr));
    std::cout << std::format("Dereferencing ptr (*)  : {}\n", *ptr);

    // Modifying the underlying value through the pointer:
    *ptr = 99;
    std::cout << std::format("New value of target    : {}\n", target); // Prints 99
}
```

#### 10.3.2 Pointer Arithmetic: Scaling by Type Width

When you add an integer offset $k$ to a pointer `ptr` of type `T*`, the
CPU does **not** add $k$ raw bytes to the address. Instead, it scales
the addition by the physical byte size of the underlying type:

$$\text{Effective Address} = \text{ptr} + (k \times \text{sizeof}(T))$$

``` cpp
#include <iostream>
#include <format>

int main() {
    std::int32_t numbers[4] = {100, 200, 300, 400};
    std::int32_t* p = numbers; // Implicit decay: p points to numbers[0]

    std::cout << std::format("Base Address p     : {}\n", static_cast<void*>(p));
    std::cout << std::format("Value at *p        : {}\n", *p);       // 100

    // Advance pointer by 1 element (adds 4 bytes, since sizeof(int32_t) == 4)
    p = p + 1;
    std::cout << std::format("Address (p + 1)    : {}\n", static_cast<void*>(p));
    std::cout << std::format("Value at *(p + 1)  : {}\n", *p);       // 200

    // Array indexing syntax is purely syntactic sugar for pointer arithmetic:
    // numbers[i] is identical to *(numbers + i)
    std::cout << std::format("numbers[2] via []  : {}\n", numbers[2]);
    std::cout << std::format("numbers[2] via *   : {}\n", *(numbers + 2));
}
```

#### 10.3.3 References versus Pointers

Modern C++ provides **References** (`T&`) alongside pointers. A
reference is an immutable alias for an already-existing object:

``` cpp
int original = 100;
int& ref = original; // ref IS original. No separate dereference needed.

ref = 250;           // original is now 250.
```

#### Core Semantic Distinctions:

- **Nullability:** A pointer can be `nullptr` (pointing to nothing). A
  reference *must* bind to a valid instance upon initialisation and can
  never be re-bound.
- **Syntax:** Pointers require dereferencing operators (`*ptr` or
  `ptr->field`). References use natural value syntax (`ref` or
  `ref.field`).
- **Idiom:** In Modern C++, prefer passing function arguments by
  `const T&` to avoid expensive copies without sacrificing type safety
  or dealing with null checks. Use pointers only when optionality
  (`nullptr`) or pointer arithmetic is explicitly required.

### 10.4 Memory Hazards, Alignment, and Vulnerabilities

When programmers manage memory manually without rigorous abstraction,
they expose systems to severe bugs.

#### 10.4.1 The Classic Hazards

1.  **Memory Leak:** Allocating dynamic memory with `new` and losing the
    pointer before calling `delete`. The memory remains claimed by the
    process, inaccessible and unrecyclable, until process termination.
2.  **Dangling Pointer (Use-After-Free):** Deallocating a block of
    memory, but retaining a pointer to it. When that memory is
    subsequently reallocated for other data, writing through the
    dangling pointer corrupts program state.
3.  **Double Free:** Calling `delete` twice on the same memory address,
    which can corrupt the free-list structures inside the runtime
    allocator.
4.  **Buffer Overflow:** Writing past the allocated boundary of an array
    on the stack or heap, overwriting adjacent variables, function
    return addresses, or heap metadata.

#### 10.4.2 Structural Alignment and Padding

CPUs do not read memory one byte at a time; they fetch memory in 32-bit
(4-byte) or 64-bit (8-byte) words. If a 64-bit integer resides at an odd
address (e.g., `0x0003`), the CPU must execute two separate memory read
cycles and stitch the halves together with shift operations, causing a
performance penalty.

To prevent this, the compiler automatically inserts invisible **Padding
Bytes** between struct fields so that every type aligns to a multiple of
its natural width (`alignof(T)`).

Consider this struct:

``` cpp
struct UnoptimisedStruct {
    char   a;      // 1 byte
    // 7 bytes of compiler padding inserted here!
    double b;      // 8 bytes (must align to 8-byte boundary)
    int    c;      // 4 bytes
    // 4 bytes of tail padding inserted here!
}; // Total size: 24 bytes (despite only 13 bytes of payload!)

struct OptimisedStruct {
    double b;      // 8 bytes (Offset 0)
    int    c;      // 4 bytes (Offset 8)
    char   a;      // 1 byte  (Offset 12)
    // 3 bytes of tail padding
}; // Total size: 16 bytes!
```

``` cpp
#include <iostream>
#include <format>

int main() {
    std::cout << std::format("Size of UnoptimisedStruct: {} bytes\n", sizeof(UnoptimisedStruct));
    std::cout << std::format("Size of OptimisedStruct  : {} bytes\n", sizeof(OptimisedStruct));
}
```

*Rule of thumb:* Order member variables from largest alignment
requirement down to smallest to minimise wasteful padding.

### 10.5 The Sovereign Idiom: RAII (Resource Acquisition Is Initialisation)

In C, if an error occurs halfway through a function, the programmer must
remember to clean up every open file, mutex, socket, and memory
allocation manually:

``` c
// The brittle C approach:
void legacy_process() {
    int* data = (int*)malloc(1024 * sizeof(int));
    if (step_one_fails()) {
        free(data); // Easy to forget!
        return;
    }
    if (step_two_fails()) {
        free(data); // Duplicate cleanup code!
        return;
    }
    free(data);
}
```

C++ solves this problem through **RAII (Resource Acquisition Is
Initialisation)**.

The core philosophy of RAII (fig. 3) is:

1.  **Acquire resources during construction (`Constructor`):** Bind the
    lifetime of a resource (heap memory, socket, file descriptor)
    strictly to the lifetime of an automatic stack-allocated object.
2.  **Release resources during destruction (`Destructor`):** When the
    stack-allocated object falls out of scope for *any* reason (normal
    return, early `break`, or an unhandled C++ exception), its
    **destructor is deterministically executed by the runtime**,
    releasing the resource instantly.

<figure id="fig:owner">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIwIDAgODUwIDI4MCIgd2lkdGg9IjEwMCUiIGhlaWdodD0iMjgwIiB4bWxu
cz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciPgogIDxkZWZzPgogICAgPHN0eWxlPgogICAg
ICA6cm9vdCB7CiAgICAgICAgLS1zdmctYmc6IHZhcigtLWJnLCAjZmZmZmZmKTsKICAgICAgICAt
LXN2Zy1mZzogdmFyKC0tZmcsICMwMDAwMDApOwogICAgICAgIC0tc3ZnLXRpbnQtMS1mYWxsYmFj
azogI2NjY2NjYzsgLyogMjAlIGJsYWNrICsgODAlIHdoaXRlICovCiAgICAgICAgLS1zdmctdGlu
dC0yLWZhbGxiYWNrOiAjOTk5OTk5OyAvKiA0MCUgYmxhY2sgKyA2MCUgd2hpdGUgKi8KICAgICAg
ICAtLXN2Zy10aW50LTE6IGNvbG9yLW1peChpbiBzcmdiLCB2YXIoLS1zdmctZmcpIDIwJSwgdmFy
KC0tc3ZnLWJnKSA4MCUpOwogICAgICAgIC0tc3ZnLXRpbnQtMjogY29sb3ItbWl4KGluIHNyZ2Is
IHZhcigtLXN2Zy1mZykgNDAlLCB2YXIoLS1zdmctYmcpIDYwJSk7CiAgICAgIH0KICAgICAgdGV4
dCAgIHsgZm9udC1mYW1pbHk6ICJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIsIG1vbm9zcGFj
ZTsgfQogICAgPC9zdHlsZT4KICAgIDxmaWx0ZXIgaWQ9InJhaWktc2hhZG93IiB4PSItMTAlIiB5
PSItMTAlIiB3aWR0aD0iMTIwJSIgaGVpZ2h0PSIxMjAlIj4KICAgICAgPGZlRHJvcFNoYWRvdyBk
eD0iMiIgZHk9IjIiIHN0ZERldmlhdGlvbj0iMyIgZmxvb2Qtb3BhY2l0eT0iMC4xMiIvPgogICAg
PC9maWx0ZXI+CiAgPC9kZWZzPgogIDwhLS0gU2NvcGUgQm94IENvbnRhaW5lciAtLT4KICA8cmVj
dCB4PSI1MCIgeT0iMjUiIHdpZHRoPSI3NTAiIGhlaWdodD0iMjMwIiByeD0iOCIgZmlsbD0idmFy
KC0tc3ZnLXRpbnQtMiwgdmFyKC0tc3ZnLXRpbnQtMi1mYWxsYmFjaykpIiBzdHJva2U9InZhcigt
LXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIgZmlsdGVyPSJ1cmwoI3JhaWktc2hhZG93KSIvPgog
IDx0ZXh0IHg9IjcwIiB5PSI1NSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZp
bGw9InZhcigtLXN2Zy1mZykiPnsgLy8gRW5jbG9zaW5nIFNjb3BlIEVudGVyZWQ8L3RleHQ+CiAg
PCEtLSBTdGVwIDE6IENvbnN0cnVjdG9yIC0tPgogIDxyZWN0IHg9Ijc1IiB5PSI3NSIgd2lkdGg9
IjMyMCIgaGVpZ2h0PSI2NSIgcng9IjYiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2
Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9
IjEuNSIvPgogIDx0ZXh0IHg9IjkwIiB5PSI5OCIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9
ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiPjEuIE9iamVjdCBJbnN0YW50aWF0aW9uIChTdGFj
ayk8L3RleHQ+CiAgPHRleHQgeD0iOTAiIHk9IjExOCIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFy
KC0tc3ZnLWZnKSI+JiM4NTk0OyBDb25zdHJ1Y3RvciBhY3F1aXJlcyBIZWFwIGJ1ZmZlcjwvdGV4
dD4KICA8IS0tIFN0ZXAgMjogVXNlIC0tPgogIDxyZWN0IHg9IjQxNSIgeT0iNzUiIHdpZHRoPSIz
NjAiIGhlaWdodD0iNjUiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmct
dGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIx
LjUiLz4KICA8dGV4dCB4PSI0MzAiIHk9Ijk4IiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0i
Ym9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+Mi4gRXhlY3V0aW9uICZhbXA7IFByb2Nlc3Npbmc8
L3RleHQ+CiAgPHRleHQgeD0iNDMwIiB5PSIxMTgiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigt
LXN2Zy1mZykiPiYjODU5NDsgRXhjZXB0aW9ucywgZWFybHkgcmV0dXJucywgY2FsY3VsYXRpb25z
PC90ZXh0PgogIDwhLS0gU3RlcCAzOiBEZXN0cnVjdG9yIC0tPgogIDxyZWN0IHg9Ijc1IiB5PSIx
NTUiIHdpZHRoPSI3MDAiIGhlaWdodD0iNTUiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0x
LCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ry
b2tlLXdpZHRoPSIxLjUiLz4KICA8dGV4dCB4PSI5MCIgeT0iMTc3IiBmb250LXNpemU9IjE0IiBm
b250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+My4gU2NvcGUgRXhpdDogRGV0
ZXJtaW5pc3RpYyBEZXN0cnVjdGlvbjwvdGV4dD4KICA8dGV4dCB4PSI5MCIgeT0iMTk1IiBmb250
LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj4mIzg1OTQ7IERlc3RydWN0b3IgZmlyZXMg
YXV0b21hdGljYWxseTogcmVsZWFzZXMgaGVhcCBtZW1vcnkgd2l0aG91dCBtYW51YWwgZnJlZSgp
PC90ZXh0PgogIDx0ZXh0IHg9IjcwIiB5PSIyNDAiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0
PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIj59IC8vIEVuY2xvc2luZyBTY29wZSBFeGl0ZWQ8
L3RleHQ+Cjwvc3ZnPg==
" alt="Figure 3: Scoped Ownership of Heap Variables" />
<figcaption>Figure 3: Scoped Ownership of Heap Variables</figcaption>
</figure>

#### 10.5.1 The Rule of Five

When an RAII class manually manages a raw dynamic resource, you must
explicitly declare the lifecycle operations known as the **Rule of
Five**:

1.  **Destructor:** To free the resource.
2.  **Copy Constructor:** To perform a deep copy of the resource.
3.  **Copy Assignment Operator:** To safely duplicate into an existing
    instance.
4.  **Move Constructor:** To transfer ownership of the resource without
    copying.
5.  **Move Assignment Operator:** To transfer ownership and discard the
    old resource.

### 10.6 Smart Pointers in Modern C++

In Modern C++, you should almost never write raw `new` or `delete` in
application code. The standard library provides zero-cost, type-safe
**Smart Pointer** abstractions inside `<memory>`.

#### 10.6.1 `std::unique_ptr<T>`: Sole, Exclusive Ownership

`std::unique_ptr` models unique, non-shared ownership of a heap
resource. It cannot be copied; it can only be **moved**. When the
`unique_ptr` falls out of scope, it deletes the object it wraps.

*Runtime Overhead:* **Zero.** A `std::unique_ptr` compiles down to the
exact same assembly as a raw pointer.

``` cpp
#include <memory>
#include <iostream>
#include <format>

struct SensorNode {
    int id;
    SensorNode(int node_id) : id(node_id) { 
        std::cout << std::format("SensorNode {} online.\n", id); 
    }
    ~SensorNode() { 
        std::cout << std::format("SensorNode {} destroyed.\n", id); 
    }
    void read_telemetry() const {
        std::cout << std::format("Node {} transmitting data.\n", id);
    }
};

void run_node_lifecycle() {
    // std::make_unique safely allocates on the heap
    std::unique_ptr<SensorNode> node = std::make_unique<SensorNode>(101);
    node->read_telemetry();

    // std::unique_ptr<SensorNode> clone = node; // COMPILE ERROR: Cannot copy unique resource!
    
    // Transfer ownership using std::move
    std::unique_ptr<SensorNode> owner = std::move(node);
    
    if (!node) {
        std::cout << "Original pointer is now empty (nullptr).\n";
    }
    owner->read_telemetry();
} // 'owner' goes out of scope -> SensorNode 101 is deleted deterministically!
```

#### 10.6.2 `std::shared_ptr<T>`: Shared Ownership via Reference Counting

When multiple independent subsystems need access to the same heap object
without knowing who will finish with it last, use `std::shared_ptr`.

A `std::shared_ptr` stores two pointers: one to the managed object, and
one to an atomic **Control Block** holding the **Reference Count**.
Every time a copy is made, the reference count increments. When an
instance goes out of scope, the reference count decrements. When the
count drops to zero, the managed object is destroyed.

``` cpp
#include <memory>
#include <iostream>
#include <format>

void inspect_node(std::shared_ptr<SensorNode> node_ref) {
    std::cout << std::format("Inside function. Ref count: {}\n", node_ref.use_count());
}

int main() {
    std::shared_ptr<SensorNode> root = std::make_shared<SensorNode>(202);
    std::cout << std::format("Initial Ref count : {}\n", root.use_count()); // 1

    {
        std::shared_ptr<SensorNode> alias = root; // Shared ownership
        std::cout << std::format("Nested Ref count  : {}\n", root.use_count()); // 2
        inspect_node(root);                                                    // 3
        std::cout << std::format("Post-call count   : {}\n", root.use_count()); // 2
    } // 'alias' destroyed -> count drops back to 1

    std::cout << std::format("Final Ref count   : {}\n", root.use_count()); // 1
} // 'root' destroyed -> count hits 0 -> SensorNode 202 is deleted.
```

#### 10.6.3 `std::weak_ptr<T>`: Breaking Circular Reference Deadlocks

If two `shared_ptr` objects hold shared pointers to each other, their
reference counts will never reach zero, causing a permanent memory leak.
A `std::weak_ptr` observes a `shared_ptr` without incrementing the
reference count, and can be (tempoarily) invoked to create a
`shared_ptr` for dereferencing within a sub-scope.

### 10.7 Experimental Implementation: Building Dynamic Linear Memory Structures in Modern C++

To master how pointers, dynamic heap management, and RAII interact, we
will construct two foundational data structures from scratch:

- lst. 1 defines a class `DynamicArray<T>`: a resizable, contiguous heap
  array implementing raw memory allocation, pointer indexing, the Rule
  of Five, and custom iterators.
- lst. 2 implements class `ContiguousMatrix<T>`: a cache-optimised 2D
  matrix structure demonstrating flat memory addressing and spatial
  locality.
- lst. 3 evaluates the performance of both of these classes.

Why does memory layout matter for performance? Because of CPU cache
lines.

When the CPU reads a single byte from memory, it pulls an entire 64-byte
**Cache Line** into L1 cache. If you represent a 2D matrix as an array
of pointers to individual dynamic rows (`int**`), every single row is
scattered at random addresses on the heap, producing severe cache
misses.

A high-performance systems implementation allocates one **single, flat
contiguous block of heap memory** of size
$(\text{Rows} \times \text{Cols})$ and translates 2D coordinates into
linear 1D memory indices:

$$\text{Index}(r, c) = (r \times \text{Cols}) + c$$

Listing 1: The Custom Dynamic Array (DynamicArray.hpp)

In [ ]:
#ifndef DYNAMIC_ARRAY_HPP
#define DYNAMIC_ARRAY_HPP

#include <cstddef>
#include <stdexcept>
#include <utility>
#include <algorithm>
#include <format>
#include <iostream>

namespace sysmem {

template <typename T>
class DynamicArray {
public:
    // =========================================================================
    // Construction and Destruction (RAII)
    // =========================================================================
    
    // Default constructor: creates an empty array with zero initial capacity
    DynamicArray() noexcept : m_data(nullptr), m_size(0), m_capacity(0) {}

    // Sized constructor: allocates capacity up front
    explicit DynamicArray(std::size_t initial_capacity)
        : m_data(initial_capacity > 0 ? new T[initial_capacity] : nullptr),
          m_size(0),
          m_capacity(initial_capacity) {}

    // Destructor: guarantees clean reclamation of heap buffer
    ~DynamicArray() noexcept {
        release();
    }

    // =========================================================================
    // The Rule of Five: Deep Copy and Move Semantics
    // =========================================================================

    // 1. Copy Constructor: Deep-copies elements from other
    DynamicArray(const DynamicArray& other)
        : m_data(other.m_capacity > 0 ? new T[other.m_capacity] : nullptr),
          m_size(other.m_size),
          m_capacity(other.m_capacity) 
    {
        for (std::size_t i = 0; i < m_size; ++i) {
            m_data[i] = other.m_data[i];
        }
    }

    // 2. Copy Assignment Operator: Copy-and-Swap Idiom for strong exception safety
    DynamicArray& operator=(const DynamicArray& other) {
        if (this != &other) {
            DynamicArray temp(other);
            swap(temp);
        }
        return *this;
    }

    // 3. Move Constructor: Steals the pointer from temporary, leaving it empty
    DynamicArray(DynamicArray&& other) noexcept
        : m_data(std::exchange(other.m_data, nullptr)),
          m_size(std::exchange(other.m_size, 0)),
          m_capacity(std::exchange(other.m_capacity, 0)) {}

    // 4. Move Assignment Operator
    DynamicArray& operator=(DynamicArray&& other) noexcept {
        if (this != &other) {
            release();
            m_data = std::exchange(other.m_data, nullptr);
            m_size = std::exchange(other.m_size, 0);
            m_capacity = std::exchange(other.m_capacity, 0);
        }
        return *this;
    }

    // =========================================================================
    // Element Access & Pointer Operations
    // =========================================================================

    [[nodiscard]] std::size_t size() const noexcept { return m_size; }
    [[nodiscard]] std::size_t capacity() const noexcept { return m_capacity; }
    [[nodiscard]] bool empty() const noexcept { return m_size == 0; }

    // Direct unchecked indexing via pointer arithmetic
    T& operator[](std::size_t index) noexcept {
        return *(m_data + index);
    }

    const T& operator[](std::size_t index) const noexcept {
        return *(m_data + index);
    }

    // Checked bounds-safe access
    T& at(std::size_t index) {
        if (index >= m_size) {
            throw std::out_of_range(std::format("DynamicArray index {} out of bounds (size: {})", index, m_size));
        }
        return *(m_data + index);
    }

    const T& at(std::size_t index) const {
        if (index >= m_size) {
            throw std::out_of_range(std::format("DynamicArray index {} out of bounds (size: {})", index, m_size));
        }
        return *(m_data + index);
    }

    // Direct access to the raw underlying pointer
    [[nodiscard]] T* data() noexcept { return m_data; }
    [[nodiscard]] const T* data() const noexcept { return m_data; }

    // =========================================================================
    // Growth and Memory Modification
    // =========================================================================

    void push_back(const T& value) {
        if (m_size == m_capacity) {
            grow(m_capacity == 0 ? 2 : m_capacity * 2);
        }
        m_data[m_size++] = value;
    }

    void push_back(T&& value) {
        if (m_size == m_capacity) {
            grow(m_capacity == 0 ? 2 : m_capacity * 2);
        }
        m_data[m_size++] = std::move(value);
    }

    void reserve(std::size_t new_capacity) {
        if (new_capacity > m_capacity) {
            grow(new_capacity);
        }
    }

    void swap(DynamicArray& other) noexcept {
        std::swap(m_data, other.m_data);
        std::swap(m_size, other.m_size);
        std::swap(m_capacity, other.m_capacity);
    }

    // Pointer-based Iterators (Enables range-based for loops: for(auto& x : arr))
    using iterator = T*;
    using const_iterator = const T*;

    iterator begin() noexcept { return m_data; }
    iterator end() noexcept { return m_data + m_size; }
    const_iterator begin() const noexcept { return m_data; }
    const_iterator end() const noexcept { return m_data + m_size; }

private:
    void grow(std::size_t target_capacity) {
        T* new_buffer = new T[target_capacity];
        
        // Move existing elements over to new memory allocation
        for (std::size_t i = 0; i < m_size; ++i) {
            new_buffer[i] = std::move(m_data[i]);
        }

        // Deallocate old heap memory and repoint
        delete[] m_data;
        m_data = new_buffer;
        m_capacity = target_capacity;
    }

    void release() noexcept {
        delete[] m_data;
        m_data = nullptr;
        m_size = 0;
        m_capacity = 0;
    }

    T*          m_data{nullptr};
    std::size_t m_size{0};
    std::size_t m_capacity{0};
};

} // namespace sysmem

#endif // DYNAMIC_ARRAY_HPP

Listing 2: Cache Locality and The Contiguous Matrix
(ContiguousMatrix.hpp)

In [ ]:
#ifndef CONTIGUOUS_MATRIX_HPP
#define CONTIGUOUS_MATRIX_HPP

#include <cstddef>
#include <vector>
#include <memory>
#include <stdexcept>
#include <format>
#include <span>

namespace sysmem {

template <typename T>
class ContiguousMatrix {
public:
    ContiguousMatrix(std::size_t rows, std::size_t cols, const T& initial_val = T{})
        : m_rows(rows),
          m_cols(cols),
          m_data(std::make_unique<T[]>(rows * cols))
    {
        if (rows == 0 || cols == 0) {
            throw std::invalid_argument("Matrix dimensions must be non-zero.");
        }
        for (std::size_t i = 0; i < (m_rows * m_cols); ++i) {
            m_data[i] = initial_val;
        }
    }

    // High-speed 2D indexing operator (Row-Major Ordering)
    [[nodiscard]] T& operator()(std::size_t row, std::size_t col) noexcept {
        return m_data[(row * m_cols) + col];
    }

    [[nodiscard]] const T& operator()(std::size_t row, std::size_t col) const noexcept {
        return m_data[(row * m_cols) + col];
    }

    [[nodiscard]] std::size_t rows() const noexcept { return m_rows; }
    [[nodiscard]] std::size_t cols() const noexcept { return m_cols; }

    // Modern C++20 View of a single row via std::span
    [[nodiscard]] std::span<T> get_row_span(std::size_t row) {
        if (row >= m_rows) {
            throw std::out_of_range("Row index out of range.");
        }
        return std::span<T>(&m_data[row * m_cols], m_cols);
    }

    // Direct access to contiguous buffer
    [[nodiscard]] T* raw_data() noexcept { return m_data.get(); }
    [[nodiscard]] const T* raw_data() const noexcept { return m_data.get(); }

private:
    std::size_t           m_rows;
    std::size_t           m_cols;
    std::unique_ptr<T[]>  m_data; // Pure RAII ownership of flat heap buffer
};

} // namespace sysmem

#endif // CONTIGUOUS_MATRIX_HPP

Listing 3: Demonstration and Memory Profiling Driver (main.cpp)

In [ ]:
// #include "DynamicArray.hpp"
// #include "ContiguousMatrix.hpp"
#include <iostream>
#include <string>
#include <chrono>

struct Tracer {
    std::string name;
    Tracer() : name("Default") { std::cout << "  [Tracer Constructor: " << name << "]\n"; }
    Tracer(std::string n) : name(std::move(n)) { std::cout << "  [Tracer Constructor: " << name << "]\n"; }
    ~Tracer() { std::cout << "  [Tracer Destructor: " << name << "]\n"; }
    Tracer(const Tracer& o) : name(o.name + " (Copy)") { std::cout << "  [Tracer Copy: " << name << "]\n"; }
    Tracer(Tracer&& o) noexcept : name(std::move(o.name)) { 
        o.name = "Empty (Moved)";
        std::cout << "  [Tracer Move: " << name << "]\n"; 
    }
    Tracer& operator=(const Tracer& o) {
        name = o.name + " (Copy Assign)";
        std::cout << "  [Tracer Copy Assign: " << name << "]\n";
        return *this;
    }
    Tracer& operator=(Tracer&& o) noexcept {
        name = std::move(o.name);
        o.name = "Empty (Moved)";
        std::cout << "  [Tracer Move Assign: " << name << "]\n";
        return *this;
    }
};

int main() {
    std::cout << "========================================================\n";
    std::cout << "1. DynamicArray Lifecycle and RAII Verification\n";
    std::cout << "========================================================\n";
    {
        sysmem::DynamicArray<Tracer> array;
        std::cout << "Pushing Tracer Alpha...\n";
        array.push_back(Tracer("Alpha"));

        std::cout << "Pushing Tracer Beta...\n";
        array.push_back(Tracer("Beta"));

        std::cout << "Array size: " << array.size() << ", Capacity: " << array.capacity() << "\n";
        std::cout << "Exiting inner scope -> All dynamic memory must destruct now:\n";
    }

    std::cout << "\n========================================================\n";
    std::cout << "2. Memory Alignment and Pointer Traversal\n";
    std::cout << "========================================================\n";
    sysmem::DynamicArray<int> numbers;
    for (int i = 1; i <= 5; ++i) {
        numbers.push_back(i * 10);
    }

    std::cout << "Pointer-based iteration via iterators (*p):\n";
    for (int val : numbers) {
        std::cout << val << " ";
    }
    std::cout << "\n";

    std::cout << "\n========================================================\n";
    std::cout << "3. Contiguous Matrix vs Cache Locality\n";
    std::cout << "========================================================\n";
    constexpr std::size_t DIM = 2048;
    std::cout << std::format("Instantiating {}x{} Matrix ({} Megabytes)...\n", 
                             DIM, DIM, (DIM * DIM * sizeof(double)) / (1024 * 1024));

    sysmem::ContiguousMatrix<double> matrix(DIM, DIM, 1.0);

    // Row-major traversal (Cache-Friendly: reading adjacent memory addresses)
    auto t0 = std::chrono::high_resolution_clock::now();
    double row_sum = 0.0;
    for (std::size_t r = 0; r < DIM; ++r) {
        for (std::size_t c = 0; c < DIM; ++c) {
            row_sum += matrix(r, c);
        }
    }
    auto t1 = std::chrono::high_resolution_clock::now();
    auto row_duration = std::chrono::duration_cast<std::chrono::milliseconds>(t1 - t0).count();
    std::cout << std::format("Row-major sum completed in  : {} ms (Sum: {})\n", row_duration, row_sum);

    // Column-major traversal (Cache-Hostile: jumping stride of DIM across memory)
    auto t2 = std::chrono::high_resolution_clock::now();
    double col_sum = 0.0;
    for (std::size_t c = 0; c < DIM; ++c) {
        for (std::size_t r = 0; r < DIM; ++r) {
            col_sum += matrix(r, c);
        }
    }
    auto t3 = std::chrono::high_resolution_clock::now();
    auto col_duration = std::chrono::duration_cast<std::chrono::milliseconds>(t3 - t2).count();
    std::cout << std::format("Column-major sum completed in: {} ms (Sum: {})\n", col_duration, col_sum);
    std::cout << std::format("Performance difference: Contiguous row access is {:.2f}x faster!\n", 
                             static_cast<double>(col_duration) / row_duration);

    return 0;
}

### 10.8 Summary

1.  **Virtual Address Space:** Your program runs within a simulated
    address space isolated by the operating system and MMU.
2.  **The Stack is Automatic and Fast:** Stack allocations cost a single
    CPU register update (`RSP`), provide high cache locality, but have
    strict size limits ($1\text{--}8\text{ MB}$).
3.  **The Heap is Dynamic and Flexible:** The heap accommodates
    gigabytes of dynamically sized objects, but requires manual
    lifecycle management or smart pointer ownership.
4.  **Pointers are Addresses:** Pointer arithmetic scales by the size of
    the underlying type ($\text{sizeof}(T)$). References provide safe,
    non-null aliases.
5.  **Alignment Dictates Layout:** Structs insert invisible padding
    bytes to keep member types aligned to their natural CPU word
    boundaries.
6.  **RAII is the Heart of C++:** Never leave resource deallocation to
    manual operations. Bind lifetime to stack objects and use
    `std::unique_ptr` and `std::shared_ptr` to ensure exception-safe,
    deterministic destruction.

### 10.9 Programming Challenges

#### Exercise 10.1: Structure Layout, Padding, and Alignment Optimisation

**Context and Theory:**

Modern 64-bit microprocessors do not fetch data from physical memory one
byte at a time. The Memory Management Unit (MMU) and memory controllers
transfer data across 64-bit (8-byte) or 128-bit data buses. To achieve
single-cycle memory reads, types must reside at memory addresses that
are integer multiples of their natural size—a property known as
**Natural Alignment**:

$$\text{Address}(x) \equiv 0 \pmod{\text{alignof}(T)}$$

If a 4-byte `std::int32_t` is placed at an odd memory address (e.g.,
`0x1001`), the CPU must execute two separate memory read cycles, perform
bitwise shifts, and merge the halves to reconstruct the integer.

To prevent this performance penalty, compilers automatically inject
invisible **Internal Padding Bytes** between structure fields and **Tail
Padding Bytes** at the end of structures to ensure that arrays of the
structure maintain alignment for every element.

The padding added after an offset can be calculated as:

$$\text{Padding} = (-\text{Offset}) \ \& \ (\text{alignof}(T) - 1)$$

Careless ordering of member variables can cause an object’s physical
memory footprint to balloon to more than double the size of its payload.

**Specification:**

Design an alignment inspection utility and optimise a telemetry data
packet structure:

1.  Analyse `UnoptimisedPacket`: calculate its theoretical memory
    footprint and compiler padding.
2.  Reorder its fields into `OptimisedPacket` to achieve the absolute
    minimal size permissible by C++ alignment rules without altering its
    semantic data types.
3.  Write a template function `print_layout<T>()` using `sizeof()`,
    `alignof()`, and `offsetof()` to output a visual map of member byte
    offsets.

``` cpp
#include <cstdint>
#include <cstddef>
#include <iostream>

struct UnoptimisedPacket {
    std::uint8_t  flag_a;      // 1 byte
    std::uint64_t timestamp;   // 8 bytes (requires 8-byte alignment)
    std::uint16_t sensor_id;   // 2 bytes
    std::uint32_t payload_len; // 4 bytes (requires 4-byte alignment)
    bool          is_active;   // 1 byte
};

struct OptimisedPacket {
    std::uint64_t timestamp;   // Offset 0 (8B)
    std::uint32_t payload_len; // Offset 8 (4B)
    std::uint16_t sensor_id;   // Offset 12 (2B)
    std::uint8_t  flag_a;      // Offset 14 (1B)
    bool          is_active;   // Offset 15 (1B)
};                             // Total: 16 bytes (Zero padding!)
```

**Verification Tasks:**

- **Task 1 (Memory Footprint Reduction):** Assert via `static_assert`
  that `sizeof(UnoptimisedPacket) == 24` bytes (containing 8 bytes of
  padding), whereas `sizeof(OptimisedPacket) == 16` bytes with
  `alignof(OptimisedPacket) == 8`.
- **Task 2 (Field Offset Verification):** Verify programmatically using
  `offsetof` that in `OptimisedPacket`,
  `offsetof(OptimisedPacket, timestamp) == 0`,
  `offsetof(OptimisedPacket, payload_len) == 8`, and
  `offsetof(OptimisedPacket, is_active) == 15`.
- **Task 3 (Contiguous Array Stride):** Create an array of 1,000,000
  instances of both structures. Measure the memory consumption delta.
  Verify that `OptimisedPacket` saves exactly 8 Megabytes of RAM.

#### Exercise 10.2: Pointer Arithmetic, Array Strides, and Address Traversal

**Context and Theory:**

In C++, a pointer is an unsigned integer holding a virtual memory
address, augmented with compile-time type semantics.

When an integer offset $k$ is added to a pointer `ptr` of type `T*`, the
physical byte address emitted by the compiler is scaled by the size of
the underlying type:

$$\text{Effective Address} = \text{reinterpret\_cast}\langle\text{uintptr\_t}\rangle(\text{ptr}) + (k \times \text{sizeof}(T))$$

Array subscript notation is syntactic sugar for pointer arithmetic:

$$\text{arr}[i] \equiv *(\text{arr} + i)$$

Understanding this scaling is essential when working with raw byte
buffers (`std::byte*`), network frame parsing, and memory-mapped
hardware interfaces. Reinterpreting a multi-byte integer as a sequence
of raw bytes reveals the host architecture’s **Endianness**
(Little-Endian vs Big-Endian).

**Specification:**

Write a memory examination and manipulation utility:

1.  Implement `hex_dump(const void* ptr, std::size_t bytes)`: prints the
    physical memory addresses and hexadecimal byte values of any object.
2.  Implement
    `raw_byte_copy(void* dest, const void* src, std::size_t bytes)`
    without using standard library routines (`memcpy`), relying purely
    on `std::byte*` pointer arithmetic and dereferencing.
3.  Implement an endianness inspector that takes a `std::uint32_t`
    literal `0x12345678` and identifies whether the machine stores the
    Least Significant Byte (`0x78`) at the lowest memory address
    (Little-Endian) or highest (Big-Endian).

``` cpp
#include <cstddef>
#include <cstdint>
#include <format>
#include <iostream>

void raw_byte_copy(void* dest, const void* src, std::size_t bytes) noexcept {
    auto* d = static_cast<std::byte*>(dest);
    const auto* s = static_cast<const std::byte*>(src);
    for (std::size_t i = 0; i < bytes; ++i) {
        *(d + i) = *(s + i); // Pointer arithmetic scaling by sizeof(std::byte) == 1
    }
}
```

**Verification Tasks:**

- **Task 1 (Address Stride Scaling):** Create an array of three
  `std::int64_t` values. Print `&arr[0]`, `&arr[1]`, and `&arr[2]`.
  Verify mathematically that the difference between adjacent addresses
  is exactly `8` bytes (`sizeof(int64_t)`).
- **Task 2 (Bit-Level Copy Verification):** Copy a complex struct
  containing integers, floats, and character arrays using
  `raw_byte_copy`. Verify that every member in the destination matches
  the source.
- **Task 3 (Endianness Detection):** Run the endianness inspector on the
  host environment. Verify that on standard x86-64 or Apple Silicon
  architectures, byte 0 of `0x12345678` reads `0x78`, proving
  Little-Endian byte ordering.

#### Exercise 10.3: Deep Copying, Move Semantics, and the Rule of Five

**Context and Theory:**

When an object manages a raw resource on the dynamic heap (such as a
pointer allocated via `new[]`), using default compiler-generated copy
operations causes undefined behaviour (see fig. 4):

<figure id="fig:shallow-copy">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2MjguNTYiIGhl
aWdodD0iNTkuODcyIiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDYyOC41NiA1OS44NzIi
PgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5z
dmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13
aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJv
a2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0
ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBJ
b3NldmthIENoYXJvbiBNb25vIE1lZGl1bSwgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTRweDsK
fQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJhbnNw
YXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoKLnN2
Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsKICBm
aWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmlsbCB7
CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewogIG1h
cmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cgewog
IG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlhbW9u
ZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2Vk
X2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuZW5k
X21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2JvYiAu
c3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0KCi5z
dmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Blbl9j
aXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1z
dGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29wZW5f
Y2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2Ig
LnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNiaWdf
b3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJyb3ci
IHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1h
cmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdv
biBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8
bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIi
IG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZl
cnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3BvbHln
b24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAwIDgg
OCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3Jp
ZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0i
MiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9
Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJX
aWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAg
ICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNs
ZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3Qm94
PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0
PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5
PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogIDwv
ZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNjI4LjU2
IiBoZWlnaHQ9IjU5Ljg3MiI+PC9yZWN0PgogIDxwb2x5Z29uIHBvaW50cz0iMjc5LjM2MDAyLDE3
LjQ2MDAwMSAyODYuMzQ0LDIwLjk1MiAyNzkuMzYwMDIsMjQuNDQ0IiBjbGFzcz0iZmlsbGVkIj48
L3BvbHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjk1LjA3NCIgeT0iMjQu
NDQ0IiA+WzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMDkuMDQyMDIi
IHk9IjI0LjQ0NCIgPkhFQVA8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MzQzLjk2MiIgeT0iMjQuNDQ0IiA+QlVGRkVSPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjM5Mi44NSIgeT0iMjQuNDQ0IiA+XTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSIxLjc0NiIgeT0iMTAuNDc2IiA+T2JqZWN0IEE6IFsgbV9kYXRhOiAweDU1
NTUgXTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0MDYuODE4MDIiIHk9
IjI0LjQ0NCIgPihPbmx5IE9ORSBwaHlzaWNhbCBhbGxvY2F0aW9uISk8L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMS43NDYiIHk9IjM4LjQxMjAwMyIgPk9iamVjdCBCOiBb
IG1fZGF0YTogMHg1NTU1IF08L3RleHQ+CiAgPGc+CiAgICA8bGluZSB4MT0iMjE2LjUwNDAxIiB5
MT0iNi45ODQiIHgyPSIyNDcuOTMyIiB5Mj0iNi45ODQiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgog
ICAgPGxpbmUgeDE9IjI0Ny45MzIiIHkxPSI2Ljk4NCIgeDI9IjI0Ny45MzIiIHkyPSIzNC45MjAw
MDIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjI0Ny45MzIiIHkxPSIyMC45
NTIiIHgyPSIyNzkuMzYwMDIiIHkyPSIyMC45NTIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAg
PGxpbmUgeDE9IjIxNi41MDQwMSIgeTE9IjM0LjkyMDAwMiIgeDI9IjI0Ny45MzIiIHkyPSIzNC45
MjAwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDwvZz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 4: Default Copy: Shallow Copy (Pointer
Duplication)</figcaption>
</figure>

When `Object B` falls out of scope, its destructor frees the heap
buffer. When `Object A` subsequently falls out of scope, it attempts to
free the same memory, triggering a catastrophic **Double Free** crash.
Furthermore, any operations performed through `Object A` before
destruction represent **Use-After-Free** hazards.

To safely encapsulate dynamic memory, an RAII class must implement the
**Rule of Five**:

1.  **Destructor:** Releases the heap allocation.
2.  **Copy Constructor:** Allocates a *new* heap buffer and deep-copies
    elements.
3.  **Copy Assignment Operator:** Safely duplicates data into an
    existing object (typically using the *Copy-and-Swap* idiom).
4.  **Move Constructor:** Transfers pointer ownership from a temporary
    object without allocation, leaving the source in a valid, empty
    state (`nullptr`).
5.  **Move Assignment Operator:** Releases existing memory, then
    transfers ownership.

**Specification:**

Implement `SafeIntBuffer`, an RAII wrapper over a dynamically allocated
array of integers:

- Manage a raw `int* m_buffer` and `std::size_t m_size`.
- Implement all operations of the Rule of Five.
- Provide an active allocation counter tracking the total number of heap
  buffers currently alive.
- Use `std::exchange` to safely nullify sources during move operations.

``` cpp
#include <utility>
#include <algorithm>
#include <cstddef>

class SafeIntBuffer {
public:
    explicit SafeIntBuffer(std::size_t size) 
        : m_buffer(size ? new int[size]() : nullptr), m_size(size) {}
    ~SafeIntBuffer() noexcept { delete[] m_buffer; }

    SafeIntBuffer(const SafeIntBuffer& other) // Deep Copy
        : m_buffer(other.m_size ? new int[other.m_size] : nullptr), m_size(other.m_size) {
        std::copy_n(other.m_buffer, m_size, m_buffer);
    }
    SafeIntBuffer(SafeIntBuffer&& other) noexcept // Move: zero allocation
        : m_buffer(std::exchange(other.m_buffer, nullptr)), m_size(std::exchange(other.m_size, 0)) {}

    SafeIntBuffer& operator=(SafeIntBuffer other) noexcept { // Copy-and-Swap
        swap(*this, other);
        return *this;
    }
    friend void swap(SafeIntBuffer& a, SafeIntBuffer& b) noexcept {
        std::swap(a.m_buffer, b.m_buffer);
        std::swap(a.m_size, b.m_size);
    }
private:
    int* m_buffer{nullptr};
    std::size_t m_size{0};
};
```

**Verification Tasks:**

- **Task 1 (Deep Copy Independence):** Create `SafeIntBuffer a(10)`.
  Initialise elements to `100`. Copy-construct `b = a`. Mutate
  `b[0] = 999`. Verify that `a[0]` remains `100` and
  `a.data() != b.data()`.
- **Task 2 (Move Semantics Zero-Allocation):** Create
  `SafeIntBuffer a(1'000'000)`. Record the heap pointer address.
  Move-construct `b = std::move(a)`. Verify that `b.data()` holds the
  original address, `a.data() == nullptr`, and zero allocations
  occurred.
- **Task 3 (Self-Assignment Safety):** Execute `a = a`. Verify that the
  object remains valid, memory is not freed prematurely, and no memory
  leaks occur.

#### Exercise 10.4: Constructing an Industrial-Grade `UniquePtr<T>` from Scratch

**Context and Theory:**

`std::unique_ptr<T>` models strict, **exclusive ownership** of a
dynamically allocated heap resource. It encapsulates a raw pointer,
guaranteeing that:

1.  Exactly one `unique_ptr` owns the resource at any given moment.
2.  The resource is deterministically freed when the `unique_ptr` goes
    out of scope.
3.  Ownership can be transferred via **Move Semantics**, but can *never*
    be copied.

Crucially, `unique_ptr` is a **Zero-Cost Abstraction**. It stores only a
single pointer with no reference counts, metadata headers, or
indirection tables. In compiled assembly under `-O2` or `-O3`, calling
`unique_ptr->method()` produces the exact same machine code as
dereferencing a raw pointer (`ptr->method()`).

**Specification:**

Implement `CustomUniquePtr<T>` from scratch:

- Delete the copy constructor and copy assignment operator (`= delete`).
- Implement the move constructor and move assignment operator.
- Overload `operator*`, `operator->`, and explicit `operator bool()` for
  null checks.
- Implement member functions `get()`, `release()` (surrenders ownership
  returning raw pointer), and `reset(T* new_ptr = nullptr)`.
- Provide a standalone factory function
  `make_custom_unique<T>(Args&&... args)` supporting perfect forwarding.

``` cpp
#include <utility>

template <typename T>
class CustomUniquePtr {
public:
    constexpr CustomUniquePtr() noexcept : m_ptr(nullptr) {}
    explicit CustomUniquePtr(T* ptr) noexcept : m_ptr(ptr) {}
    ~CustomUniquePtr() noexcept { delete m_ptr; }

    CustomUniquePtr(const CustomUniquePtr&) = delete;
    CustomUniquePtr& operator=(const CustomUniquePtr&) = delete;

    CustomUniquePtr(CustomUniquePtr&& other) noexcept 
        : m_ptr(std::exchange(other.m_ptr, nullptr)) {}

    CustomUniquePtr& operator=(CustomUniquePtr&& other) noexcept {
        if (this != &other) {
            reset(std::exchange(other.m_ptr, nullptr));
        }
        return *this;
    }

    [[nodiscard]] T* get() const noexcept { return m_ptr; }
    [[nodiscard]] T* release() noexcept { return std::exchange(m_ptr, nullptr); }
    void reset(T* ptr = nullptr) noexcept {
        delete std::exchange(m_ptr, ptr);
    }
    T& operator*() const noexcept { return *m_ptr; }
    T* operator->() const noexcept { return m_ptr; }
    explicit operator bool() const noexcept { return m_ptr != nullptr; }
private:
    T* m_ptr;
};
```

**Verification Tasks:**

- **Task 1 (Deterministic Deletion):** Create a struct
  `DestructionTracker` that increments a global counter upon
  destruction. Wrap an instance in `CustomUniquePtr`. Allow it to fall
  out of scope. Verify that the counter increments immediately.
- **Task 2 (Move Ownership Transfer):** Construct `ptrA` managing a
  resource. Move it to `ptrB`. Verify that `ptrA.get() == nullptr`,
  `ptrB` points to the valid object, and the resource is not destroyed
  during the transfer.
- **Task 3 (Release Prevents Deletion):** Call `ptr.release()`. Allow
  `ptr` to fall out of scope. Verify that the underlying object was
  *not* destroyed, and manually delete the raw pointer to prevent a
  leak.

#### Exercise 10.5: Reference-Counted Shared Ownership (`SharedPtr<T>`)

**Context and Theory:**

When multiple independent systems need shared access to the same heap
object without knowing which subsystem will finish using it last,
exclusive ownership cannot be used.

`std::shared_ptr<T>` implements **Shared Ownership via Reference
Counting**.

A `shared_ptr` instance does not point solely to the managed object; it
maintains a second pointer to an independent heap-allocated structure
called the **Control Block** (see fig. 5). The control block stores:

1.  `use_count`: The number of active `shared_ptr` owners.
2.  `weak_count`: The number of `weak_ptr` observers.
3.  Allocator and Custom Deleter metadata.

<figure id="fig:ref-counting">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI0MCAyMCA3NzAgMTYwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIxNjAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiwgbW9ub3Nw
YWNlOyB9CiAgICA8L3N0eWxlPgogIDwvZGVmcz4KICA8cmVjdCB4PSI1MCIgeT0iMzAiIHdpZHRo
PSIxODAiIGhlaWdodD0iNjAiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1z
dmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIvPgogIDx0ZXh0IHg9
IjYwIiB5PSI1NSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigt
LXN2Zy1mZykiPnNoYXJlZF9wdHIgQTwvdGV4dD4KICA8dGV4dCB4PSI2MCIgeT0iNzUiIGZvbnQt
c2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPm9ial9wdHIgfCBjdHJsX3B0cjwvdGV4dD4K
ICA8cmVjdCB4PSI1MCIgeT0iMTEwIiB3aWR0aD0iMTgwIiBoZWlnaHQ9IjYwIiByeD0iNiIgZmls
bD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9
InZhcigtLXN2Zy1mZykiLz4KICA8dGV4dCB4PSI2MCIgeT0iMTM1IiBmb250LXNpemU9IjE0IiBm
b250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+c2hhcmVkX3B0ciBCPC90ZXh0
PgogIDx0ZXh0IHg9IjYwIiB5PSIxNTUiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1m
ZykiPmN0cmxfcHRyIHwgb2JqX3B0cjwvdGV4dD4KICA8cmVjdCB4PSIzNjAiIHk9IjUwIiB3aWR0
aD0iMjAwIiBoZWlnaHQ9IjEwMCIgcng9IjYiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigt
LXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIi8+CiAgPHRleHQg
eD0iMzc1IiB5PSI3NSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZh
cigtLXN2Zy1mZykiPkNvbnRyb2wgQmxvY2s8L3RleHQ+CiAgPHRleHQgeD0iMzc1IiB5PSIxMDUi
IGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPnVzZV9jb3VudDogMjwvdGV4dD4K
ICA8dGV4dCB4PSIzNzUiIHk9IjEyNSIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZn
KSI+d2Vha19jb3VudDogMDwvdGV4dD4KICA8cmVjdCB4PSI2NjAiIHk9IjUwIiB3aWR0aD0iMTQw
IiBoZWlnaHQ9IjEwMCIgcng9IjYiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10
aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIi8+CiAgPHRleHQgeD0iNjc1
IiB5PSI3NSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2
Zy1mZykiPk1hbmFnZWQgVDwvdGV4dD4KICA8dGV4dCB4PSI2NzUiIHk9IjEwNSIgZm9udC1zaXpl
PSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+RGF0YSBQYXlsb2FkPC90ZXh0PgogIDxwYXRoIGQ9
Ik0gMjMwIDYwIEMgNDAwIDIwLCA2MDAgMjAsIDY2MCA4MCIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcp
IiBmaWxsPSJub25lIiBzdHJva2Utd2lkdGg9IjEuNSIgc3Ryb2tlLWRhc2hhcnJheT0iMiwyIi8+
CiAgPHBhdGggZD0iTSAyMzAgMTUwIEMgNTAwIDE4MCwgNjAwIDE2MCwgNjYwIDExNSIgc3Ryb2tl
PSJ2YXIoLS1zdmctZmcpIiBmaWxsPSJub25lIiBzdHJva2Utd2lkdGg9IjEuNSIgc3Ryb2tlLWRh
c2hhcnJheT0iMiwyIi8+CiAgPHBhdGggZD0iTSAyMzAgNzUgTCAzNjAgODUiIHN0cm9rZT0idmFy
KC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIyIi8+CiAgPHBhdGggZD0iTSAyMzAgMTM1IEwgMzYw
IDExNSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiLz4KPC9zdmc+
" alt="Figure 5: Reference Counting and Control Block" />
<figcaption>Figure 5: Reference Counting and Control Block</figcaption>
</figure>

When a copy of `shared_ptr` is made, `use_count` is incremented. When an
instance is destroyed, `use_count` is decremented. When `use_count`
reaches zero, the managed object is destroyed.

**Specification:**

Implement `SimpleSharedPtr<T>`:

- Create a heap-allocated `ControlBlock` storing
  `std::size_t ref_count`.
- Implement the Copy Constructor and Copy Assignment operator to
  increment `ref_count`.
- Implement the Destructor: decrement `ref_count`; if zero, delete both
  the managed object and the control block.
- Implement `use_count()` to report active shared owners.

``` cpp
template <typename T>
class SimpleSharedPtr {
    struct ControlBlock { std::size_t use_count{1}; };
    T* m_ptr{nullptr};
    ControlBlock* m_ctrl{nullptr};
public:
    explicit SimpleSharedPtr(T* ptr) 
        : m_ptr(ptr), m_ctrl(ptr ? new ControlBlock() : nullptr) {}
    ~SimpleSharedPtr() { release(); }

    SimpleSharedPtr(const SimpleSharedPtr& o) noexcept : m_ptr(o.m_ptr), m_ctrl(o.m_ctrl) {
        if (m_ctrl) ++m_ctrl->use_count;
    }
    void release() noexcept {
        if (m_ctrl && --m_ctrl->use_count == 0) {
            delete m_ptr;
            delete m_ctrl;
        }
    }
    [[nodiscard]] std::size_t use_count() const noexcept { return m_ctrl ? m_ctrl->use_count : 0; }
    T& operator*() const noexcept { return *m_ptr; }
};
```

**Verification Tasks:**

- **Task 1 (Reference Count Lifecycle):** Construct `sp1`. Verify
  `sp1.use_count() == 1`. In an inner scope, copy-construct `sp2 = sp1`
  and `sp3 = sp2`. Verify `sp1.use_count() == 3`. Exit the inner scope.
  Verify `sp1.use_count() == 1`.
- **Task 2 (Shared Destruction):** Attach a destruction tracker to
  `SimpleSharedPtr`. Create three copies. Reset two. Verify the object
  is not destroyed. Reset the final pointer; verify the object is
  deleted immediately.
- **Task 3 (Move Semantics Avoid Count Overhead):** Move-construct
  `sp2 = std::move(sp1)`. Verify that `use_count` remains `1`,
  `sp1.use_count() == 0`, and no increment/decrement cycles occurred.

#### Exercise 10.6: Breaking Cyclic Reference Deadlocks with Weak Pointers

**Context and Theory:**

While `std::shared_ptr` provides automatic memory management, it has a
major vulnerability: **Cyclic References**.

If Object A holds a `std::shared_ptr` to Object B, and Object B holds a
`std::shared_ptr` to Object A, a reference cycle is formed:

$$\text{use\_count}(A) = 1, \quad \text{use\_count}(B) = 1$$

When external pointers to A and B fall out of scope, both reference
counts decrement from 2 to 1. Because neither count drops to 0, neither
object is ever destroyed. This produces a permanent, undetectable
**Memory Leak**.

To break reference cycles, C++ provides **`std::weak_ptr<T>`**. A
`weak_ptr` observes an object managed by a `shared_ptr` without
incrementing its `use_count`. Because a `weak_ptr` does not guarantee
the object remains alive, it cannot be dereferenced directly; it must
first be promoted to a valid `std::shared_ptr` via `.lock()`.

**Specification:**

Design a bidirectional hierarchical graph representing a file system or
organisational tree:

1.  Define a `Node` class containing `std::string name`,
    `std::vector<std::shared_ptr<Node>> children`, and a pointer back to
    its `parent`.
2.  Demonstrate that using `std::shared_ptr<Node>` for the `parent`
    pointer causes a cyclic reference leak when a child node is added.
3.  Fix the leak by changing the `parent` pointer to
    `std::weak_ptr<Node>`.
4.  Implement a method `std::string get_full_path() const` that
    traverses up the tree using `parent.lock()`.

``` cpp
#include <memory>
#include <string>
#include <vector>

struct Node {
    std::string name;
    std::weak_ptr<Node> parent; // Non-owning back-reference prevents cycles!
    std::vector<std::shared_ptr<Node>> children;

    Node(std::string n) : name(std::move(n)) {}
    
    void add_child(std::shared_ptr<Node> child, std::shared_ptr<Node> self) {
        child->parent = self; // Binds weak pointer without incrementing use_count
        children.push_back(std::move(child));
    }
};
```

**Verification Tasks:**

- **Task 1 (Cycle Memory Leak Reproduction):** Implement the tree using
  `std::shared_ptr` for `parent`. Instantiate a parent and child inside
  a nested scope. Count active destructors on scope exit; verify that
  neither node is destroyed, causing a leak.
- **Task 2 (Weak Pointer Leak Resolution):** Switch `parent` to
  `std::weak_ptr`. Run the same test. Verify that both the parent and
  child destructors execute on scope exit.
- **Task 3 (Safe Traversal via `.lock()`):** Create a 3-level tree
  (`Root -> Folder -> File`). Call `get_full_path()` on `File`. Verify
  it returns `"/Root/Folder/File"`. Reset `Root`. Verify that subsequent
  calls to `parent.lock()` yield `nullptr`.

#### Exercise 10.7: Deterministic Lifecycle via Placement New and Explicit Destruction

**Context and Theory:**

Standard dynamic allocation (`new T(...)`) performs two distinct
operations:

1.  It allocates raw memory on the heap via `operator new(sizeof(T))`.
2.  It executes the constructor of `T` in-place inside that allocated
    memory block.

Similarly, `delete ptr` executes `T`’s destructor, then releases memory
via `operator delete`.

In high-performance systems (such as custom containers, game engines,
and embedded kernels), allocating memory dynamically for every new
object is too slow. Engineers pre-allocate raw, uninitialised memory
buffers and separate memory allocation from object construction using
**Placement New**:

``` cpp
new (storage_address) T(constructor_args...);
```

When an object constructed via placement new reaches the end of its
lifecycle, you **must not** call `delete`, as the memory was not
allocated via standard `new`. Instead, you must **explicitly invoke the
destructor**:

``` cpp
ptr->~T();
```

**Specification:**

Implement a type-safe, single-element storage container
`OptionalStorage<T>` (a minimal implementation of `std::optional`):

- Allocate inline, uninitialised memory using a properly aligned byte
  buffer: `alignas(alignof(T)) std::byte m_storage[sizeof(T)];`
- Implement `emplace(Args&&... args)`: constructs an instance of `T`
  inside `m_storage` using placement new.
- Implement `reset()`: explicitly calls the destructor
  `reinterpret_cast<T*>(m_storage)->~T()` and marks the storage empty.
- Enforce that constructing or destroying `OptionalStorage<T>`
  guarantees correct lifecycle execution for complex non-trivial types.

``` cpp
#include <new>
#include <cstddef>
#include <utility>

template <typename T>
class OptionalStorage {
    alignas(alignof(T)) std::byte m_storage[sizeof(T)];
    bool m_has_value{false};
public:
    OptionalStorage() = default;
    ~OptionalStorage() { reset(); }

    template <typename... Args>
    void emplace(Args&&... args) {
        reset();
        new (m_storage) T(std::forward<Args>(args)...); // Placement New
        m_has_value = true;
    }
    void reset() noexcept {
        if (m_has_value) {
            reinterpret_cast<T*>(m_storage)->~T(); // Explicit Destructor Invocation
            m_has_value = false;
        }
    }
    T& value() { return *reinterpret_cast<T*>(m_storage); }
};
```

**Verification Tasks:**

- **Task 1 (Construction Deferral):** Wrap `DestructionTracker` inside
  `OptionalStorage`. Verify that constructing the container executes
  zero constructor calls until `emplace()` is invoked.
- **Task 2 (Explicit Destruction):** Call `emplace()`, then call
  `reset()`. Verify that the destructor is executed while the underlying
  `OptionalStorage` stack object remains alive.
- **Task 3 (Alignment Invariant):** Instantiate
  `OptionalStorage<double>`. Verify that
  `reinterpret_cast<uintptr_t>(m_storage) % alignof(double) == 0`.

#### Exercise 10.8: High-Performance Arena (Bump) Allocator

**Context and Theory:**

Standard general-purpose heap allocators (`malloc` / `new`) are designed
to handle arbitrary allocation sizes and random deallocation orders. To
manage this flexibility, they maintain internal bookkeeping structures
(free-lists, boundary tags, bin trees) protected by thread locks. As a
result, allocations can cost dozens or hundreds of clock cycles and
produce memory fragmentation.

In compilers, game render loops, and request-response servers, memory
allocations follow an **Arena Lifespan**: many small objects are
allocated rapidly during a single computational phase, and all of them
can be freed simultaneously at the end of the phase.

A **Bump Allocator (Arena)** pre-allocates a single contiguous block of
memory. Allocations simply advance an internal pointer by the requested
byte count plus alignment padding (as shown in fig. 6):

$$\text{Next Address} = (\text{Current Offset} + \text{Alignment} - 1) \ \& \ \sim(\text{Alignment} - 1)$$

- Allocation cost: **$O(1)$** (a pointer addition and mask).
- Deallocation cost: **$O(1)$** (resetting the offset pointer back to
  zero).

<figure id="fig:bump-allocator">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI0MCAyMCA3NzAgMTAwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIxMDAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiwgbW9ub3Nw
YWNlOyB9CiAgICA8L3N0eWxlPgogIDwvZGVmcz4KICA8cmVjdCB4PSI1MCIgeT0iMzAiIHdpZHRo
PSI3NTAiIGhlaWdodD0iNTAiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1z
dmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIvPgogIDx0ZXh0IHg9
IjY1IiB5PSI2MCIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+QWxsb2MgMSAo
NjRCKTwvdGV4dD4KICA8bGluZSB4MT0iMTgwIiB5MT0iMzAiIHgyPSIxODAiIHkyPSI4MCIgc3Ry
b2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjEiLz4KICA8dGV4dCB4PSIyMTAiIHk9
IjYwIiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5BbGxvYyAyICgxMjhCKTwv
dGV4dD4KICA8bGluZSB4MT0iMzUwIiB5MT0iMjAiIHgyPSIzNTAiIHkyPSI5MCIgc3Ryb2tlPSJ2
YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiLz4KICA8dGV4dCB4PSIzNDMiIHk9IjExMCIg
Zm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiPuKG
kSBtX29mZnNldCAoQnVtcCBQb2ludGVyKTwvdGV4dD4KICA8dGV4dCB4PSI0MDAiIHk9IjYwIiBm
b250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5VbmFsbG9jYXRlZCBQcmUtUmVzZXJ2
ZWQgQnVmZmVyIChSZW1haW5pbmcgQ2FwYWNpdHkpPC90ZXh0Pgo8L3N2Zz4=
" alt="Figure 6: Bump Allocator Pointer Movement" />
<figcaption>Figure 6: Bump Allocator Pointer Movement</figcaption>
</figure>

**Specification:**

Implement `ArenaAllocator`:

- Allocate a contiguous backing buffer of size $N$ bytes on the heap
  upon construction.
- Implement
  `void* allocate(std::size_t bytes, std::size_t alignment = alignof(std::max_align_t))`:
  calculates alignment padding, bumps the internal offset, and returns
  the aligned pointer.
- Throw `std::bad_alloc` if the allocation exceeds capacity.
- Implement `void reset() noexcept`: resets the allocation offset to
  zero without releasing the backing buffer.
- Destructor frees the entire backing buffer.

``` cpp
#include <cstddef>
#include <cstdint>
#include <new>

class ArenaAllocator {
    std::byte*  m_buffer;
    std::size_t m_capacity;
    std::size_t m_offset{0};
public:
    explicit ArenaAllocator(std::size_t cap) 
        : m_buffer(new std::byte[cap]), m_capacity(cap) {}
    ~ArenaAllocator() { delete[] m_buffer; }

    [[nodiscard]] void* allocate(std::size_t bytes, std::size_t alignment) {
        const auto current_addr = reinterpret_cast<std::uintptr_t>(m_buffer + m_offset);
        const auto aligned_addr = (current_addr + alignment - 1) & ~(alignment - 1);
        const auto new_offset = (aligned_addr - reinterpret_cast<std::uintptr_t>(m_buffer)) + bytes;
        if (new_offset > m_capacity) throw std::bad_alloc();
        m_offset = new_offset;
        return reinterpret_cast<void*>(aligned_addr);
    }
    void reset() noexcept { m_offset = 0; }
};
```

**Verification Tasks:**

- **Task 1 (Alignment Guarantee):** Request 100 consecutive allocations
  of varying sizes and alignments (1, 2, 4, 8, 16 bytes). Assert that
  every returned pointer satisfies
  `reinterpret_cast<uintptr_t>(ptr) % align == 0`.
- **Task 2 (Capacity Limit Verification):** Instantiate an arena with a
  1024-byte capacity. Allocate 1000 bytes successfully. Attempt to
  allocate an additional 100 bytes; verify that `std::bad_alloc` is
  thrown.
- **Task 3 (Allocation Speed Benchmark):** Compare allocating 1,000,000
  small objects via `new` against the Arena allocator. Verify that the
  Arena allocator is at least $5\times$ faster.

#### Exercise 10.9: Fixed-Size Block (Slab) Allocator with an Intrusive Free-List

**Context and Theory:**

While an Arena allocator offers fast allocations, it cannot free
individual objects; it can only reset the entire memory space at once.

When a program frequently creates and destroys objects of an identical,
fixed size (e.g., node allocations in linked lists, trees, or network
packet descriptors), a **Slab Allocator** provides $O(1)$ allocation
*and* $O(1)$ individual deallocation without fragmentation.

A slab pre-allocates an array of fixed-sized blocks. To track
unallocated slots without consuming extra memory, it uses an **Intrusive
Free-List**: when a block is unallocated, its memory is cast to a
pointer pointing to the next available free block (see fig. 7).

<figure id="fig:fixed-allocator">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2NzAuNDY0IiBo
ZWlnaHQ9IjU5Ljg3MiIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA2NzAuNDY0IDU5Ljg3
MiI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxNHB4
Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFu
c3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgou
c3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewog
IGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxs
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJr
ZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5l
bmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVu
X2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2Vy
LXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Bl
bl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Jp
Z19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJv
dyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIg
bWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5
Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAg
IDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0i
MiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJl
dmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9s
eWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAg
OCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBv
cmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiBy
PSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBp
ZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtl
cldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4K
ICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2ly
Y2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdC
b3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWln
aHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIg
Y3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAg
PC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI2NzAu
NDY0IiBoZWlnaHQ9IjU5Ljg3MiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjEuNzQ2IiB5PSIxMC40NzYiID5GcmVlPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjM2LjY2NiIgeT0iMTAuNDc2IiA+TGlzdDo8L3RleHQ+CiAgPGxpbmUgeDE9IjE2
Ny42MTYiIHkxPSI2Ljk4NCIgeDI9IjE4MS41ODQiIHkyPSI2Ljk4NCIgY2xhc3M9InNvbGlkIj48
L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIxODEuNTg0LDMuNDkyIDE4OC41NjgwMSw2Ljk4NCAx
ODEuNTg0LDEwLjQ3NiIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjE5Ny4yOTgiIHk9IjEwLjQ3NiIgPls8L3RleHQ+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjExLjI2NiIgeT0iMTAuNDc2IiA+QmxvY2s8L3RleHQ+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjUzLjE3MDAxIiB5PSIxMC40NzYiID5BPC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI2Ny4xMzgiIHk9IjEwLjQ3NiIg
Pl08L3RleHQ+CiAgPGxpbmUgeDE9IjI3OS4zNjAwMiIgeTE9IjYuOTg0IiB4Mj0iMjkzLjMyOCIg
eTI9IjYuOTg0IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSIyOTUuMDc0IiB5PSIxMC40NzYiID5wb2ludHM8L3RleHQ+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iMzQzLjk2MiIgeT0iMTAuNDc2IiA+dG88L3RleHQ+CiAgPGxpbmUg
eDE9IjM2My4xNjgiIHkxPSI2Ljk4NCIgeDI9IjM3Ny4xMzYwMiIgeTI9IjYuOTg0IiBjbGFzcz0i
c29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjM3Ny4xMzYwMiwzLjQ5MiAzODQuMTIw
MDMsNi45ODQgMzc3LjEzNjAyLDEwLjQ3NiIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM5Mi44NSIgeT0iMTAuNDc2IiA+WzwvdGV4dD4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0MDYuODE4MDIiIHk9IjEwLjQ3NiIgPkJs
b2NrPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQ0OC43MjIwMiIgeT0i
MTAuNDc2IiA+QjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0NjIuNjki
IHk9IjEwLjQ3NiIgPl08L3RleHQ+CiAgPGxpbmUgeDE9IjQ3NC45MTIwMiIgeTE9IjYuOTg0IiB4
Mj0iNDg4Ljg4IiB5Mj0iNi45ODQiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjQ5MC42MjYiIHk9IjEwLjQ3NiIgPnBvaW50czwvdGV4dD4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1MzkuNTE0MDQiIHk9IjEwLjQ3NiIgPnRvPC90
ZXh0PgogIDxsaW5lIHgxPSI1NTguNzIwMDMiIHkxPSI2Ljk4NCIgeDI9IjU3Mi42ODgiIHkyPSI2
Ljk4NCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSI1NzIuNjg4LDMu
NDkyIDU3OS42NzIsNi45ODQgNTcyLjY4OCwxMC40NzYiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdv
bj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1ODguNDAyMDQiIHk9IjEwLjQ3NiIg
Pm51bGxwdHI8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNzguNTciIHk9
IjEwLjQ3NiIgPm1fZnJlZV9oZWFkPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjEuNzQ2IiB5PSIyNC40NDQiID5XaGVuIEJsb2NrIEEgaXMgYWxsb2NhdGVkLCBtX2ZyZWVf
aGVhZCBhZHZhbmNlcyB0byBCbG9jayBCLjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSIxLjc0NiIgeT0iMzguNDEyMDAzIiA+V2hlbiBCbG9jayBBIGlzIGZyZWVkLCBpdHMg
bWVtb3J5IHN0b3JlcyB0aGUgb2xkIG1fZnJlZV9oZWFkLCBhbmQgbV9mcmVlX2hlYWQgcG9pbnRz
IGJhY2sgdG8gQSE8L3RleHQ+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 7: Intrusive Free-List Allocation</figcaption>
</figure>

**Specification:**

Implement `FixedSlabAllocator<T, BlockCount>`:

- Pre-allocate a contiguous memory block holding `BlockCount` instances
  of `T`.
- Maintain a pointer `Node* m_free_head` pointing to the first available
  slot.
- Implement `T* allocate()`: pops the head node from the free list in
  $O(1)$ time.
- Implement `void deallocate(T* ptr)`: pushes the freed block back onto
  the intrusive list in $O(1)$ time without calling `delete`.
- Ensure that `sizeof(T) >= sizeof(void*)` so the intrusive pointer fits
  inside the block.

``` cpp
#include <cstddef>
#include <new>

template <typename T, std::size_t BlockCount>
class FixedSlabAllocator {
    union Slot {
        Slot* next_free;
        alignas(alignof(T)) std::byte storage[sizeof(T)];
    };
    Slot m_pool[BlockCount];
    Slot* m_free_head{nullptr};
public:
    FixedSlabAllocator() {
        for (std::size_t i = 0; i < BlockCount - 1; ++i) 
            m_pool[i].next_free = &m_pool[i + 1];
        m_pool[BlockCount - 1].next_free = nullptr;
        m_free_head = &m_pool[0];
    }
    T* allocate() {
        if (!m_free_head) throw std::bad_alloc();
        Slot* slot = m_free_head;
        m_free_head = m_free_head->next_free;
        return reinterpret_cast<T*>(slot);
    }
    void deallocate(T* ptr) noexcept {
        auto* slot = reinterpret_cast<Slot*>(ptr);
        slot->next_free = m_free_head;
        m_free_head = slot;
    }
};
```

**Verification Tasks:**

- **Task 1 (Full Capacity Pool Recycling):** For an allocator of
  capacity 10, allocate all 10 blocks. Verify the 11th allocation throws
  `std::bad_alloc`. Deallocate 3 blocks, then allocate 3 blocks again.
  Verify all allocations succeed.
- **Task 2 (LIFO Memory Reuse):** Allocate block $A$, then block $B$.
  Deallocate $B$, then deallocate $A$. Allocate a new block $C$. Verify
  that $C$ reuses the exact memory address of $A$.
- **Task 3 (Destruction Independence):** Confirm that cycling
  allocations and deallocations does not trigger any underlying calls to
  global `malloc` or `free`.

#### Exercise 10.10: Global Memory Leak and Heap Profiler via Operator Overloads

**Context and Theory:**

Memory leaks occur when dynamic memory is allocated on the heap, but all
pointers referencing that allocation are lost before `delete` is called.
Over time, leaks exhaust available virtual memory, causing memory
paging, performance degradation, and `Out-Of-Memory (OOM)` process
termination.

In C++, developers can intercept every dynamic memory allocation across
the entire application by overloading the global `operator new` and
`operator delete` functions:

``` cpp
void* operator new(std::size_t size);
void operator delete(void* ptr) noexcept;
```

By intercepting these calls, an instrumentation harness can record
metadata for every allocation (physical address, byte size, active
allocation counts), alerting developers if any memory remains unfreed
when `main()` terminates.

**Specification:**

Design a lightweight, thread-safe dynamic memory tracking harness:

1.  Overload global `operator new` and `operator delete`.
2.  Maintain an internal `std::unordered_map<void*, std::size_t>`
    storing active heap addresses and their allocation sizes.
3.  Prevent infinite recursion by temporarily disabling tracking during
    the profiler’s own internal container allocations.
4.  Provide a report function `report_leaks()` that prints active leak
    counts, total leaked bytes, and addresses.

``` cpp
#include <new>
#include <cstdlib>
#include <iostream>
#include <format>

static bool g_tracking_enabled = false;
static std::size_t g_total_allocated = 0;
static std::size_t g_active_allocations = 0;

void* operator new(std::size_t size) {
    void* ptr = std::malloc(size);
    if (!ptr) throw std::bad_alloc();
    if (g_tracking_enabled) {
        g_total_allocated += size;
        ++g_active_allocations;
    }
    return ptr;
}

void operator delete(void* ptr) noexcept {
    if (ptr && g_tracking_enabled) {
        --g_active_allocations;
    }
    std::free(ptr);
}
```

**Verification Tasks:**

- **Task 1 (Clean Lifecycle Verification):** Enable tracking. Allocate
  an array of 100 integers and delete it. Verify that
  `g_active_allocations == 0`.
- **Task 2 (Leak Detection):** Allocate three blocks of memory on the
  heap (sizes 32, 64, and 128 bytes). Deliberately omit calling
  `delete`. Run `report_leaks()`. Verify that the profiler detects 3
  active allocations and exactly 224 leaked bytes.
- **Task 3 (Standard Library Tracking):** Initialise a
  `std::vector<int>` and push elements until it reallocates. Confirm
  that the profiler records the dynamic growth and subsequent
  destruction of the vector’s internal buffers.

#### Exercise 10.11: Hardware Cache Stride Profiler and Spatial Locality

**Context and Theory:**

When a program accesses a variable at memory address $A$, the CPU does
not fetch only that variable. It loads an entire **64-byte Cache Line**
containing address $A$ into the L1 Data Cache.

- **Spatial Locality:** If an application accesses address $A$, it is
  highly likely to access adjacent addresses ($A + 1, A + 2$) soon.
  Processing contiguous memory sequentiality exploits this, running at
  high speed because almost every access hits the L1/L2 cache.
- **Strided Traversal:** If a program accesses memory with a large step
  size (stride $S$), every access jumps to a different cache line,
  invalidating prefetchers and stalling the execution pipeline while
  waiting for data from main RAM.

$$\text{Memory Access Latency}: \text{L1 Cache} \approx 1\text{ ns} \quad \text{vs.} \quad \text{Main RAM} \approx 60\text{--}100\text{ ns}$$

Traversing memory with a stride larger than 64 bytes can degrade
performance by more than an order of magnitude.

**Specification:**

Construct a cache stride benchmarking suite:

1.  Allocate a flat, contiguous dynamic array of $64\text{ MB}$ of
    `std::int32_t` elements.

2.  Implement a summing function that traverses the array using a
    configurable stride parameter
    $S \in \{1, 2, 4, 8, 16, 32, 64, 128\}$:

    ``` cpp
    for (std::size_t i = 0; i < N; i += stride) sum += arr[i];
    ```

3.  Use high-resolution timers (`std::chrono::high_resolution_clock`) to
    measure the average time taken per memory access across varying
    strides.

4.  Output the results to observe the performance cliff that occurs when
    the stride exceeds the 64-byte cache-line threshold
    ($\text{stride} \ge 16$ integers).

``` cpp
#include <chrono>
#include <vector>
#include <span>

std::uint64_t benchmark_stride(std::span<const std::int32_t> data, std::size_t stride) {
    auto start = std::chrono::high_resolution_clock::now();
    std::int64_t sum = 0;
    for (std::size_t i = 0; i < data.size(); i += stride) {
        sum += data[i];
    }
    auto end = std::chrono::high_resolution_clock::now();
    return std::chrono::duration_cast<std::chrono::microseconds>(end - start).count();
}
```

**Verification Tasks:**

- **Task 1 (Stride-1 Baseline):** Run the benchmark with `stride = 1`.
  Verify that reading 16,000,000 contiguous integers completes rapidly
  due to automatic hardware stream prefetching.
- **Task 2 (Cache-Line Cliff Observation):** Run the benchmark for
  strides 1, 2, 4, 8, 16, and 32. Confirm that execution time per read
  increases significantly once the stride reaches 16 elements
  ($16 \times 4\text{ bytes} = 64\text{ bytes}$).
- **Task 3 (Sum Invariance):** Scale the accumulated sums mathematically
  to verify that every run processes the expected subset of elements
  correctly.

#### Exercise 10.12: Cache-Optimal Sparse Matrix via Compressed Row Storage (CRS)

**Context and Theory:**

In scientific computing, graph analysis, and physics simulations, large
matrices often contain predominantly zero entries (often $>99\%$
sparse).

Representing an $N \times N$ sparse matrix as a dense 2D array
(`double[N][N]`) consumes $O(N^2)$ memory, wasting space on zeroes.
Conversely, representing it as an array of linked lists or pointer
arrays (`double**`) introduces severe pointer-chasing and CPU cache
misses.

The industrial-standard solution is **Compressed Row Storage (CRS)**.
CRS represents an arbitrary sparse matrix using only **three contiguous
1D arrays** (see fig. 8):

1.  `values`: Stores non-zero values contiguously.
2.  `col_indices`: Stores the column index for each entry in `values`.
3.  `row_offsets`: An array of size $(\text{Rows} + 1)$ where
    `row_offsets[r]` marks the starting index in `values` for row $r$.

$$\text{Number of non-zero elements in row } r = \text{row\_offsets}[r+1] - \text{row\_offsets}[r]$$

CRS eliminates pointers entirely, ensures data is packed contiguously in
memory, and allows cache-friendly matrix-vector multiplication
($y = A \cdot x$).

<figure id="fig:compress-row">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI0MCAyMCA3ODAgMTIwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIxMjAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiwgbW9ub3Nw
YWNlOyB9CiAgICA8L3N0eWxlPgogIDwvZGVmcz4KICA8cmVjdCB4PSI1MCIgeT0iMzAiIHdpZHRo
PSIxODAiIGhlaWdodD0iMTAwIiByeD0iNiIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0t
c3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiLz4KICA8dGV4dCB4
PSI2MCIgeT0iNTUiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIo
LS1zdmctZmcpIj5EZW5zZSAzeDMgTWF0cml4PC90ZXh0PgogIDx0ZXh0IHg9IjYwIiB5PSI4MCIg
Zm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+WyA1LjAsICAwLjAsICAwLjAgXTwv
dGV4dD4KICA8dGV4dCB4PSI2MCIgeT0iOTgiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2
Zy1mZykiPlsgMC4wLCAgOC4wLCAgMC4wIF08L3RleHQ+CiAgPHRleHQgeD0iNjAiIHk9IjExNiIg
Zm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+WyAzLjAsICAwLjAsICA5LjAgXTwv
dGV4dD4KICA8cmVjdCB4PSIyODAiIHk9IjMwIiB3aWR0aD0iNTIwIiBoZWlnaHQ9IjEwMCIgcng9
IjYiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIg
c3Ryb2tlPSJ2YXIoLS1zdmctZmcpIi8+CiAgPHRleHQgeD0iMjk1IiB5PSI1NSIgZm9udC1zaXpl
PSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiPkNvbXByZXNzZWQg
Um93IFN0b3JhZ2UgKENSUyk8L3RleHQ+CiAgPHRleHQgeD0iMjk1IiB5PSI4MCIgZm9udC1zaXpl
PSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+dmFsdWVzICAgICAgOiBbIDUuMCwgOC4wLCAzLjAs
IDkuMCBdPC90ZXh0PgogIDx0ZXh0IHg9IjI5NSIgeT0iOTgiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9
InZhcigtLXN2Zy1mZykiPmNvbF9pbmRpY2VzIDogWyAgIDAsICAgMSwgICAwLCAgIDIgXTwvdGV4
dD4KICA8dGV4dCB4PSIyOTUiIHk9IjExNiIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3Zn
LWZnKSI+cm93X29mZnNldHMgOiBbICAgMCwgICAxLCAgIDIsICAgNCBdPC90ZXh0Pgo8L3N2Zz4=
" alt="Figure 8: Sparse Matrix Implementation (Compressed Row Storage)" />
<figcaption>Figure 8: Sparse Matrix Implementation (Compressed Row
Storage)</figcaption>
</figure>

**Specification:** Implement `SparseMatrixCRS<T>`: \* Use
`std::vector<T> m_values`, `std::vector<std::size_t> m_col_indices`, and
`std::vector<std::size_t> m_row_offsets`. \* Provide a constructor
taking dimensions and an initialisation list of tuples
`(row, col, value)`. \* Implement
`T operator()(std::size_t r, std::size_t c) const`: performs a binary
search or linear scan over the slice of `col_indices` bounded by
`row_offsets[r]` and `row_offsets[r+1]`. \* Implement
`std::vector<T> multiply(std::span<const T> x) const`: computes
$y = A \cdot x$ using linear streaming memory traversal.

``` cpp
#include <vector>
#include <span>

template <typename T>
class SparseMatrixCRS {
    std::size_t m_rows, m_cols;
    std::vector<T>           m_values;
    std::vector<std::size_t> m_col_indices;
    std::vector<std::size_t> m_row_offsets;
public:
    std::vector<T> multiply(std::span<const T> x) const {
        std::vector<T> y(m_rows, T{0});
        for (std::size_t r = 0; r < m_rows; ++r) {
            const auto start = m_row_offsets[r];
            const auto end   = m_row_offsets[r + 1];
            for (std::size_t i = start; i < end; ++i) {
                y[r] += m_values[i] * x[m_col_indices[i]];
            }
        }
        return y;
    }
};
```

**Verification Tasks:** \* **Task 1 (Element Lookup Correctness):**
Construct a $4 \times 4$ sparse matrix containing non-zero elements at
$(0, 0)=10$, $(1, 2)=20$, and $(3, 1)=30$. Verify that
`matrix(1, 2) == 20` and unpopulated entries (e.g., `matrix(0, 1)`)
return `0`. \* **Task 2 (Matrix-Vector Multiplication):** Multiply the
matrix from Task 1 by the vector $x = [1, 2, 3, 4]^T$. Verify that the
computed output vector matches:
$$y = [10(1), 20(3), 0, 30(2)]^T = [10, 60, 0, 60]^T$$ \* **Task 3
(Memory Footprint Reduction):** Initialise a $1000 \times 1000$ matrix
with only 500 non-zero entries. Measure the total bytes used by CRS
versus a dense $1000 \times 1000$ array. Verify that CRS reduces memory
consumption by over $95\%$.